# Extended comparison notebook: safer + more untried methods for CV R² improvement


In [1]:
# ============================================================
# Cell A1. Imports + Global Config
# ============================================================

import os
import re
import json
import math
import warnings
from copy import deepcopy
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict

import joblib
import numpy as np
import pandas as pd

from openpyxl import load_workbook
from openpyxl.utils import column_index_from_string, get_column_letter

from scipy.stats import spearmanr, loguniform, uniform, randint

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.feature_selection import VarianceThreshold, mutual_info_regression

from sklearn.model_selection import GroupKFold, GroupShuffleSplit, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

from sklearn.linear_model import Ridge, ElasticNet, ElasticNetCV, BayesianRidge, HuberRegressor
from sklearn.cross_decomposition import PLSRegression
from sklearn.kernel_ridge import KernelRidge
from sklearn.svm import SVR

warnings.filterwarnings("ignore")

# ----------------------------
# Raw import config
# ----------------------------
IMPORT_PATH = r"C:\Users\김민겸\Desktop\Metal metamaterial\Training\Total data_260503.xlsx"

RUN_TAG = datetime.now().strftime("%y%m%d_%H%M%S")
EXPORT_ROOT_NAME = f"Result_enhanced_featureaware_{RUN_TAG}"
EXPORT_ROOT_DIR = os.path.join(os.path.dirname(IMPORT_PATH), EXPORT_ROOT_NAME)

RAW_EXPORT_DIR = os.path.join(EXPORT_ROOT_DIR, "01_RawData")
MODEL_EXPORT_DIR = os.path.join(EXPORT_ROOT_DIR, "02_ModelComparison")
FINAL_BACKUP_DIR = os.path.join(EXPORT_ROOT_DIR, "03_FinalModelBackup")
FINAL_DATA_DIR = os.path.join(EXPORT_ROOT_DIR, "04_FinalSelectedDatasets")
PER_OUTPUT_DIR = os.path.join(MODEL_EXPORT_DIR, "per_output")

HEADER_MAIN_ROW = 2
HEADER_SUB_ROWS = [3, 4, 5]

DATA_ROW_START = 7
DATA_ROW_END = 204

RANGE_INPUT  = ("I",  DATA_ROW_START, "FU", DATA_ROW_END)
RANGE_OUTPUT = ("FV", DATA_ROW_START, "HL", DATA_ROW_END)

OUTPUT_COLUMNS = ["FW", "FX", "FZ", "GA", "GC", "GG", "GJ", "GZ", "HA", "HB", "HC", "HD", "HE", "HG", "HI", "HK"]

EXPORT_FORMAT_XLSX = True
EXPORT_FORMAT_CSV  = True

# ----------------------------
# Dataset config
# ----------------------------
ROUND_X_DECIMALS = 8
MIN_GROUPS_FOR_MODEL = 12
TEST_OUTPUTS = None
# 예시: TEST_OUTPUTS = ["Modulus", "Yield strength"]

# ----------------------------
# CV config
# ----------------------------
RANDOM_STATE = 42
FINAL_OUTER_SPLITS = 5
OUTER_REPEATS = 12
OUTER_TEST_SIZE = 0.22
INNER_GSS_SPLITS = 24
INNER_GSS_TEST_SIZE = 0.20
SEARCH_N_JOBS = 1

# ----------------------------
# Feature/model config
# ----------------------------
ROUGH_PREFILTER_TOPK = 20
ROUGH_PREFILTER_TOPK_BY_FAMILY = {
    "mechanical_energy": 16,
    "thermal": 20,
    "vibrational": 22,
}
FINAL_TOPK_CANDIDATES = [2, 3, 4, 6]
TOPK_BY_FAMILY = {
    "mechanical_energy": [2, 3, 4],
    "thermal": [2, 3, 4, 6],
    "vibrational": [2, 3, 4, 6],
}
MAX_FINAL_FEATURES = 6
CORR_PRUNE_THRESHOLD = 0.85

FEATURE_SCORE_WEIGHTS = {
    "spearman": 0.30,
    "pearson": 0.15,
    "mutual_info": 0.15,
    "elastic_net": 0.40,
}

MODEL_AWARE_SCORING_MODELS = ["Bayesian_Ridge", "Ridge", "PLS_Regression"]
FINAL_MODEL_NAMES = [
    "Bayesian_Ridge",
    "Ridge",
    "ElasticNet_CV",
    "Huber_Regressor",
    "PLS_Regression",
    "PCR_Ridge",
    "Kernel_Ridge_RBF",
    "SVR_RBF",
]

MODEL_COMPLEXITY_RANK = {
    "Bayesian_Ridge": 1,
    "Ridge": 2,
    "ElasticNet_CV": 3,
    "Huber_Regressor": 4,
    "PLS_Regression": 5,
    "PCR_Ridge": 6,
    "Kernel_Ridge_RBF": 7,
    "SVR_RBF": 8,
    "WeightedBlend_2": 9,
    "WeightedBlend_3": 10,
}

TARGET_TRANSFORM_CANDIDATES = ["raw", "yeo_johnson"]

INNER_SCORING_BY_FAMILY = {
    "mechanical_energy": "r2",
    "thermal": "r2",
    "vibrational": "r2",
}
SUMMARY_STD_PENALTY = 0.10
SUMMARY_SUPPORT_BONUS = 0.12
MIN_SUPPORT_SHARE_BY_FAMILY = {
    "mechanical_energy": 0.15,
    "thermal": 0.15,
    "vibrational": 0.20,
}

SELECTION_TOLERANCE_FRAC = 0.03
MIN_TRAIN_R2_GATE = 0.01
CONSENSUS_MIN_FOLD_SHARE = 0.40
ROBUST_TRIM_Z = 1.5

# ----------------------------
# Conservative feature engineering config
# ----------------------------
ENGINEERED_FEATURES_ENABLED = True
ENGINEERED_MAX_BASE_BY_FAMILY = {
    "mechanical_energy": 4,
    "thermal": 5,
    "vibrational": 5,
}
ENGINEERED_INCLUDE_SIGNED_LOG = True
ENGINEERED_INCLUDE_SQUARE = True
ENGINEERED_INCLUDE_PRODUCT_BY_FAMILY = {
    "mechanical_energy": True,
    "thermal": True,
    "vibrational": True,
}
ENGINEERED_INCLUDE_RATIO_BY_FAMILY = {
    "mechanical_energy": True,
    "thermal": True,
    "vibrational": False,
}
ENGINEERED_MAX_TOTAL_ADDED = 18

# repeated-Y handling
APPLY_REPEATED_Y_ONLY_IF_NEEDED = True

# selection objective
TOPK_PENALTY = 0.010
COMPLEXITY_PENALTY = 0.005
MIN_VALID_EVAL_SAMPLES = 3



os.makedirs(RAW_EXPORT_DIR, exist_ok=True)
os.makedirs(MODEL_EXPORT_DIR, exist_ok=True)
os.makedirs(FINAL_BACKUP_DIR, exist_ok=True)
os.makedirs(FINAL_DATA_DIR, exist_ok=True)
os.makedirs(PER_OUTPUT_DIR, exist_ok=True)

print("IMPORT_PATH      :", IMPORT_PATH)
print("EXPORT_ROOT_DIR  :", EXPORT_ROOT_DIR)
print("FINAL_BACKUP_DIR :", FINAL_BACKUP_DIR)
print("FINAL_DATA_DIR   :", FINAL_DATA_DIR)
print("FINAL_MODEL_NAMES:", FINAL_MODEL_NAMES)
print("OUTER_REPEATS    :", OUTER_REPEATS)
print("TOPK_BY_FAMILY   :", TOPK_BY_FAMILY)

# ----------------------------
# Blend / robust search config
# ----------------------------
BLEND_TOP_CANDIDATES = [2, 3]
BLEND_MIN_BASE_SCORE = 0.0
BLEND_EVAL_FOLDS = 5
TRAIN_WINSOR_CLIP = 0.02


IMPORT_PATH      : C:\Users\김민겸\Desktop\Metal metamaterial\Training\Total data_260503.xlsx
EXPORT_ROOT_DIR  : C:\Users\김민겸\Desktop\Metal metamaterial\Training\Result_enhanced_featureaware_260508_104114
FINAL_BACKUP_DIR : C:\Users\김민겸\Desktop\Metal metamaterial\Training\Result_enhanced_featureaware_260508_104114\03_FinalModelBackup
FINAL_DATA_DIR   : C:\Users\김민겸\Desktop\Metal metamaterial\Training\Result_enhanced_featureaware_260508_104114\04_FinalSelectedDatasets
FINAL_MODEL_NAMES: ['Bayesian_Ridge', 'Ridge', 'ElasticNet_CV', 'Huber_Regressor', 'PLS_Regression', 'PCR_Ridge', 'Kernel_Ridge_RBF', 'SVR_RBF']
OUTER_REPEATS    : 12
TOPK_BY_FAMILY   : {'mechanical_energy': [2, 3, 4], 'thermal': [2, 3, 4, 6], 'vibrational': [2, 3, 4, 6]}


In [2]:
# ============================================================
# Cell A2. Helper Functions - Excel / Header / Export
# ============================================================

def _ffill_horiz(values):
    out = []
    last = None
    for v in values:
        if v is None or str(v).strip() == "":
            out.append(last)
        else:
            last = str(v).strip()
            out.append(last)
    return out

def _sanitize_header(text):
    text = "" if text is None else str(text)
    text = text.replace("\n", " ").replace("\r", " ").strip()
    text = re.sub(r"\s+", " ", text)
    return text

def _make_unique(names):
    seen = Counter()
    out = []
    for n in names:
        base = n if n else "Unnamed"
        seen[base] += 1
        out.append(base if seen[base] == 1 else f"{base}__{seen[base]}")
    return out

def build_headers_from_rows(ws, start_col_letter, end_col_letter, main_row, sub_rows):
    start_col = column_index_from_string(start_col_letter)
    end_col = column_index_from_string(end_col_letter)
    col_indices = list(range(start_col, end_col + 1))

    all_rows = [main_row] + list(sub_rows)
    row_values = []
    for r in all_rows:
        vals = [ws.cell(row=r, column=c).value for c in col_indices]
        row_values.append(_ffill_horiz(vals))

    headers = []
    for i, col_idx in enumerate(col_indices):
        parts = []
        for row_vals in row_values:
            v = row_vals[i]
            if v is not None and str(v).strip() != "":
                v = _sanitize_header(v)
                if len(parts) == 0 or parts[-1] != v:
                    parts.append(v)
        if len(parts) == 0:
            parts = [f"COL_{get_column_letter(col_idx)}"]
        headers.append(" | ".join(parts))

    return _make_unique(headers), [get_column_letter(c) for c in col_indices]

def extract_range_df(ws, range_spec, header_main_row, header_sub_rows):
    start_col, start_row, end_col, end_row = range_spec
    headers, letters = build_headers_from_rows(
        ws=ws,
        start_col_letter=start_col,
        end_col_letter=end_col,
        main_row=header_main_row,
        sub_rows=header_sub_rows
    )
    start_idx = column_index_from_string(start_col)
    end_idx = column_index_from_string(end_col)

    data = []
    for r in range(start_row, end_row + 1):
        row_vals = [ws.cell(row=r, column=c).value for c in range(start_idx, end_idx + 1)]
        data.append(row_vals)

    df = pd.DataFrame(data, columns=headers)
    df.attrs["excel_letters"] = letters
    return df

def export_df(df, path_no_ext, index=False):
    if EXPORT_FORMAT_CSV:
        df.to_csv(path_no_ext + ".csv", index=index, encoding="utf-8-sig")
    if EXPORT_FORMAT_XLSX:
        df.to_excel(path_no_ext + ".xlsx", index=index)

def _to_jsonable(obj):
    if obj is None or isinstance(obj, (str, int, float, bool)):
        return obj
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    if isinstance(obj, pd.Timestamp):
        return obj.isoformat()
    if isinstance(obj, np.ndarray):
        return [_to_jsonable(x) for x in obj.tolist()]
    if isinstance(obj, pd.Series):
        if obj.index.is_unique:
            return {str(k): _to_jsonable(v) for k, v in obj.to_dict().items()}
        return [_to_jsonable(x) for x in obj.tolist()]
    if isinstance(obj, pd.DataFrame):
        return [{str(k): _to_jsonable(v) for k, v in row.items()} for row in obj.to_dict(orient="records")]
    if isinstance(obj, dict):
        return {str(k): _to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple, set)):
        return [_to_jsonable(x) for x in obj]
    if hasattr(obj, "item"):
        try:
            return _to_jsonable(obj.item())
        except Exception:
            pass
    return str(obj)

def export_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(_to_jsonable(obj), f, ensure_ascii=False, indent=2)

def safe_numeric_df(df):
    out = df.copy()
    for c in out.columns:
        out[c] = pd.to_numeric(out[c], errors="coerce")
    return out

def sanitize_filename(text, max_len=120):
    text = re.sub(r'[\\/:*?"<>|]+', "_", str(text))
    text = re.sub(r"\s+", " ", text).strip()
    return text[:max_len]

In [3]:
# ============================================================
# Cell B1. Workbook Load + Raw Range Extraction
# ============================================================

wb = load_workbook(IMPORT_PATH, data_only=True)
sheet_name = wb.sheetnames[0]
ws = wb[sheet_name]

input_raw_df = extract_range_df(ws, RANGE_INPUT, HEADER_MAIN_ROW, HEADER_SUB_ROWS)
output_raw_all_df = extract_range_df(ws, RANGE_OUTPUT, HEADER_MAIN_ROW, HEADER_SUB_ROWS)

output_letter_to_name = dict(zip(output_raw_all_df.attrs["excel_letters"], output_raw_all_df.columns))
selected_output_names = [output_letter_to_name[c] for c in OUTPUT_COLUMNS if c in output_letter_to_name]
output_raw_df = output_raw_all_df[selected_output_names].copy()

export_df(input_raw_df, os.path.join(RAW_EXPORT_DIR, "input_raw"))
export_df(output_raw_df, os.path.join(RAW_EXPORT_DIR, "output_raw_selected"))

print("Loaded workbook :", sheet_name)
print("Input shape     :", input_raw_df.shape)
print("Output shape    :", output_raw_df.shape)
print("Selected outputs:", len(selected_output_names))
display(pd.DataFrame({"output_excel_col": OUTPUT_COLUMNS, "output_name": selected_output_names}))

Loaded workbook : 총정리
Input shape     : (198, 169)
Output shape    : (198, 16)
Selected outputs: 16


,output_excel_col,output_name
0,FW,Modulus
1,FX,Com. Strength
2,FZ,APS
3,GA,AS
4,GC,Yield strength
5,GG,Densif. strength
6,GJ,Total energy
7,GZ,Thermal characteristics | Thermal conductivity...
8,HA,Thermal characteristics | h | W/m.K
9,HB,Thermal characteristics | Heating rate | °C/s


In [4]:
# ============================================================
# Cell B2. Numeric conversion + Same-X Group ID + DATA_BY_OUTPUT
# ============================================================

X_all = safe_numeric_df(input_raw_df)
Y_all = safe_numeric_df(output_raw_df)

X_all = X_all.loc[:, X_all.notna().any(axis=0)].copy()
Y_all = Y_all.loc[:, Y_all.notna().any(axis=0)].copy()

def build_group_ids_from_x(X_df, decimals=8):
    X_round = X_df.round(decimals)
    key_series = X_round.astype(str).agg("||".join, axis=1)
    codes, _ = pd.factorize(key_series, sort=False)
    return pd.Series(codes, name="GROUP_ID"), key_series.rename("X_KEY")

GROUP_ID, X_KEY = build_group_ids_from_x(X_all, decimals=ROUND_X_DECIMALS)

MASTER_DF = pd.concat([X_all, Y_all, GROUP_ID, X_KEY], axis=1)
group_size_df = GROUP_ID.value_counts().sort_index().rename("group_size").reset_index()
group_size_df.columns = ["GROUP_ID", "group_size"]

export_df(MASTER_DF, os.path.join(RAW_EXPORT_DIR, "master_numeric_with_group"))
export_df(group_size_df, os.path.join(RAW_EXPORT_DIR, "group_size_summary"))

DATA_BY_OUTPUT = {}
target_output_names = Y_all.columns.tolist()
if TEST_OUTPUTS is not None:
    target_output_names = [c for c in target_output_names if c in TEST_OUTPUTS]

summary_rows = []
for output_name in target_output_names:
    tmp = pd.concat([X_all, Y_all[[output_name]], GROUP_ID, X_KEY], axis=1)
    tmp = tmp.dropna(subset=[output_name]).reset_index(drop=True)
    if tmp.empty:
        continue

    DATA_BY_OUTPUT[output_name] = {
        "df": tmp.copy(),
        "feature_cols": X_all.columns.tolist(),
        "target_col": output_name,
        "group_col": "GROUP_ID",
        "xkey_col": "X_KEY",
    }

    summary_rows.append({
        "output_name": output_name,
        "n_rows": int(len(tmp)),
        "n_groups": int(tmp["GROUP_ID"].nunique()),
        "n_features": int(len(X_all.columns)),
    })

DATASET_SUMMARY_DF = pd.DataFrame(summary_rows).sort_values(["n_groups", "n_rows"], ascending=[False, False])
export_df(DATASET_SUMMARY_DF, os.path.join(MODEL_EXPORT_DIR, "dataset_summary_by_output"))

display(DATASET_SUMMARY_DF)
print("Prepared outputs:", len(DATA_BY_OUTPUT))

,output_name,n_rows,n_groups,n_features
12,Vibrational response | FRF (g/N) | 300-8000 Hz...,193,128,169
13,Vibrational response | FRF (g/N) | 300-3000 Hz...,193,128,169
14,Vibrational response | FRF (g/N) | 3000-6500 H...,193,128,169
15,Vibrational response | FRF (g/N) | 6500-8000 H...,193,128,169
7,Thermal characteristics | Thermal conductivity...,65,65,169
9,Thermal characteristics | Heating rate | °C/s,65,65,169
10,Thermal characteristics | Cooling rate | °C/s,65,65,169
11,Thermal characteristics | Heating Temp | °C/s,65,65,169
0,Modulus,56,56,169
1,Com. Strength,56,56,169


Prepared outputs: 16


In [5]:
# ============================================================
# Cell C1. Repeated-group target variance diagnostics
# ============================================================

diag_rows = []

for output_name, bundle in DATA_BY_OUTPUT.items():
    df = bundle["df"].copy()
    target_col = bundle["target_col"]
    group_col = bundle["group_col"]

    # numeric safety
    df[target_col] = pd.to_numeric(df[target_col], errors="coerce")
    df = df.dropna(subset=[target_col, group_col]).copy()

    if df.empty:
        continue

    grp = (
        df.groupby(group_col)[target_col]
        .agg(["count", "mean", "median", "std", "min", "max"])
        .reset_index(drop=True)
    )

    grp["std"] = grp["std"].fillna(0.0)
    grp["range"] = (grp["max"] - grp["min"]).fillna(0.0)
    grp["cv_like"] = grp["std"] / (grp["mean"].abs() + 1e-12)
    grp["cv_like"] = grp["cv_like"].replace([np.inf, -np.inf], np.nan).fillna(0.0)

    diag_rows.append({
        "output_name": output_name,
        "n_groups": int(df[group_col].nunique()),
        "n_rows": int(len(df)),
        "median_group_size": float(df.groupby(group_col).size().median()),
        "median_group_std": float(grp["std"].median()),
        "mean_group_std": float(grp["std"].mean()),
        "median_group_range": float(grp["range"].median()),
        "mean_group_cv_like": float(grp["cv_like"].mean()),
    })

GROUP_VARIANCE_DIAG_DF = pd.DataFrame(diag_rows)

if not GROUP_VARIANCE_DIAG_DF.empty:
    GROUP_VARIANCE_DIAG_DF = GROUP_VARIANCE_DIAG_DF.sort_values(
        "mean_group_std", ascending=False
    ).reset_index(drop=True)

display(GROUP_VARIANCE_DIAG_DF)
export_df(
    GROUP_VARIANCE_DIAG_DF,
    os.path.join(MODEL_EXPORT_DIR, "repeated_group_target_variance_diag")
)

def classify_output_family(output_name):
    name = str(output_name)
    if "Thermal characteristics" in name:
        return "thermal"
    if "Vibrational response" in name:
        return "vibrational"
    return "mechanical_energy"

GROUP_VARIANCE_DIAG_DF["output_family"] = (
    GROUP_VARIANCE_DIAG_DF["output_name"].map(classify_output_family)
)

display(GROUP_VARIANCE_DIAG_DF)
export_df(
    GROUP_VARIANCE_DIAG_DF,
    os.path.join(MODEL_EXPORT_DIR, "target_variance_diagnostics")
)

,output_name,n_groups,n_rows,median_group_size,median_group_std,mean_group_std,median_group_range,mean_group_cv_like
0,Vibrational response | FRF (g/N) | 6500-8000 H...,128,193,1.0,0.0,0.099135,0.0,0.112132
1,Vibrational response | FRF (g/N) | 3000-6500 H...,128,193,1.0,0.0,0.037563,0.0,0.089067
2,Vibrational response | FRF (g/N) | 300-8000 Hz...,128,193,1.0,0.0,0.025187,0.0,0.068057
3,Vibrational response | FRF (g/N) | 300-3000 Hz...,128,193,1.0,0.0,0.006979,0.0,0.094633
4,Modulus,56,56,1.0,0.0,0.000000,0.0,0.000000
5,Com. Strength,56,56,1.0,0.0,0.000000,0.0,0.000000
6,APS,56,56,1.0,0.0,0.000000,0.0,0.000000
7,AS,56,56,1.0,0.0,0.000000,0.0,0.000000
8,Yield strength,56,56,1.0,0.0,0.000000,0.0,0.000000
9,Densif. strength,56,56,1.0,0.0,0.000000,0.0,0.000000


,output_name,n_groups,n_rows,median_group_size,median_group_std,mean_group_std,median_group_range,mean_group_cv_like,output_family
0,Vibrational response | FRF (g/N) | 6500-8000 H...,128,193,1.0,0.0,0.099135,0.0,0.112132,vibrational
1,Vibrational response | FRF (g/N) | 3000-6500 H...,128,193,1.0,0.0,0.037563,0.0,0.089067,vibrational
2,Vibrational response | FRF (g/N) | 300-8000 Hz...,128,193,1.0,0.0,0.025187,0.0,0.068057,vibrational
3,Vibrational response | FRF (g/N) | 300-3000 Hz...,128,193,1.0,0.0,0.006979,0.0,0.094633,vibrational
4,Modulus,56,56,1.0,0.0,0.000000,0.0,0.000000,mechanical_energy
5,Com. Strength,56,56,1.0,0.0,0.000000,0.0,0.000000,mechanical_energy
6,APS,56,56,1.0,0.0,0.000000,0.0,0.000000,mechanical_energy
7,AS,56,56,1.0,0.0,0.000000,0.0,0.000000,mechanical_energy
8,Yield strength,56,56,1.0,0.0,0.000000,0.0,0.000000,mechanical_energy
9,Densif. strength,56,56,1.0,0.0,0.000000,0.0,0.000000,mechanical_energy


In [6]:

# ============================================================
# Cell C2. Core helper functions required by method-comparison
# ============================================================

def rankdata_average_ties(x):
    order = np.argsort(x)
    ranks = np.empty(len(x), dtype=float)
    i = 0
    while i < len(x):
        j = i
        while j + 1 < len(x) and x[order[j + 1]] == x[order[i]]:
            j += 1
        rank = 0.5 * (i + j) + 1
        ranks[order[i:j + 1]] = rank
        i = j + 1
    return ranks

def minmax_series(s):
    s = pd.Series(s).astype(float)
    if len(s) == 0 or s.nunique(dropna=False) <= 1:
        return pd.Series(np.zeros(len(s)), index=s.index)
    mn, mx = np.nanmin(s.values), np.nanmax(s.values)
    return (s - mn) / (mx - mn + 1e-12)

def pearson_corr_safe(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 3:
        return 0.0
    xs, ys = x[mask], y[mask]
    if np.std(xs) < 1e-12 or np.std(ys) < 1e-12:
        return 0.0
    return float(np.corrcoef(xs, ys)[0, 1])

def spearman_corr_safe(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 3:
        return 0.0
    xs, ys = x[mask], y[mask]
    if np.std(xs) < 1e-12 or np.std(ys) < 1e-12:
        return 0.0
    return pearson_corr_safe(rankdata_average_ties(xs), rankdata_average_ties(ys))

def repeated_group_splits(groups, n_splits=5, n_repeats=8, random_state=42, return_split_id=False, test_size=0.22):
    groups = np.asarray(groups)
    idx = np.arange(len(groups))
    for rep in range(n_repeats):
        gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state + rep)
        tr, te = next(gss.split(idx, groups=groups))
        if return_split_id:
            yield rep, rep, 0, tr, te
        else:
            yield rep, (tr, te)

def group_repeat_stats(groups):
    ser = pd.Series(groups)
    vc = ser.value_counts()
    return {
        "n_groups": int(vc.shape[0]),
        "max_group_size": int(vc.max()) if len(vc) else 0,
        "has_repeats": bool((vc > 1).any()),
        "repeat_fraction": float((vc > 1).mean()) if len(vc) else 0.0,
    }

def get_best_y_strategy_for_output(output_name, groups=None):
    fam = classify_output_family(output_name)
    if APPLY_REPEATED_Y_ONLY_IF_NEEDED and groups is not None:
        rep = group_repeat_stats(groups)
        if not rep["has_repeats"]:
            return "singleton_raw"
    if fam in ("thermal", "vibrational"):
        return "robust_trimmed_median"
    return "hard_closest_oof"

def robust_group_target(values, z=1.5):
    v = pd.Series(np.asarray(values, dtype=float)).dropna()
    if len(v) == 0:
        return np.nan
    if len(v) <= 2:
        return float(v.median())
    med = float(v.median())
    mad = float(np.median(np.abs(v - med)))
    if mad < 1e-12:
        return med
    zscore = 0.6745 * (v - med) / mad
    keep = v[np.abs(zscore) <= z]
    if len(keep) == 0:
        keep = v
    return float(keep.median())

def aggregate_by_group_firstX(X_df, y, groups, y_method="median"):
    rows, ys, gs = [], [], []
    y = np.asarray(y, dtype=float)
    groups = np.asarray(groups)
    for g in pd.unique(groups):
        idx = np.where(groups == g)[0]
        block_X = X_df.iloc[idx]
        block_y = y[idx]
        rows.append(block_X.iloc[0].copy())
        ys.append(float(np.nanmean(block_y) if y_method == "mean" else np.nanmedian(block_y)))
        gs.append(g)
    Xg = pd.DataFrame(rows).reset_index(drop=True)
    yg = pd.Series(ys, name="target").reset_index(drop=True)
    gg = pd.Series(gs, name="GROUP_ID").reset_index(drop=True)
    return Xg, yg, gg

def make_target_transformer(method):
    if method == "raw":
        return None
    if method == "yeo_johnson":
        return PowerTransformer(method="yeo-johnson", standardize=True)
    return None

def make_pipeline_and_space(model_name, n_features, random_state=42):
    max_comp = max(1, min(int(n_features), 8))
    if model_name == "Bayesian_Ridge":
        pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", BayesianRidge())])
        return pipe, {}, "grid", 1
    if model_name == "Ridge":
        pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", Ridge(random_state=random_state))])
        return pipe, {"model__alpha": loguniform(1e-4, 1e3)}, "random", 20
    if model_name == "ElasticNet_CV":
        pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", ElasticNet(max_iter=30000, random_state=random_state))])
        return pipe, {"model__alpha": loguniform(1e-4, 1e1), "model__l1_ratio": uniform(0.05, 0.90)}, "random", 24
    if model_name == "Huber_Regressor":
        pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", HuberRegressor(max_iter=3000))])
        return pipe, {"model__alpha": loguniform(1e-6, 1e-1), "model__epsilon": uniform(1.15, 0.85)}, "random", 18
    if model_name == "PLS_Regression":
        pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", PLSRegression())])
        return pipe, {"model__n_components": list(range(1, max_comp + 1))}, "grid", 1
    if model_name == "PCR_Ridge":
        pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("pca", PCA()), ("model", Ridge(random_state=random_state))])
        return pipe, {"pca__n_components": list(range(1, max_comp + 1)), "model__alpha": loguniform(1e-4, 1e3)}, "random", 24
    if model_name == "Kernel_Ridge_RBF":
        pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", KernelRidge(kernel="rbf"))])
        return pipe, {"model__alpha": loguniform(1e-4, 1e2), "model__gamma": loguniform(1e-4, 1e1)}, "random", 24
    if model_name == "SVR_RBF":
        pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", SVR(kernel="rbf"))])
        return pipe, {"model__C": loguniform(1e-2, 1e2), "model__gamma": loguniform(1e-4, 1e1), "model__epsilon": loguniform(1e-4, 1e0)}, "random", 28
    raise ValueError(model_name)

def fit_search_model(X_train, y_train, groups_train, model_name, target_transform="raw", scoring="r2", random_state=42):
    pipe, params, search_type, n_iter = make_pipeline_and_space(model_name, X_train.shape[1], random_state=random_state)
    transformer = make_target_transformer(target_transform)
    estimator = pipe if transformer is None else TransformedTargetRegressor(regressor=pipe, transformer=transformer, check_inverse=False)
    wrapped_params = params if transformer is None else {f"regressor__{k}": v for k, v in params.items()}
    n_unique_groups = len(pd.unique(groups_train))
    cv = GroupShuffleSplit(n_splits=INNER_GSS_SPLITS, test_size=INNER_GSS_TEST_SIZE, random_state=random_state) if n_unique_groups >= 4 else GroupKFold(n_splits=max(2, n_unique_groups))
    if search_type == "grid":
        search = GridSearchCV(estimator=estimator, param_grid=wrapped_params, scoring=scoring, cv=cv, n_jobs=SEARCH_N_JOBS, refit=True, error_score=np.nan)
    else:
        search = RandomizedSearchCV(estimator=estimator, param_distributions=wrapped_params, n_iter=n_iter, scoring=scoring, cv=cv, n_jobs=SEARCH_N_JOBS, refit=True, random_state=random_state, error_score=np.nan)
    search.fit(X_train, y_train, groups=groups_train)
    cvres = pd.DataFrame(search.cv_results_)
    score_col = "mean_test_score"
    std_col = "std_test_score"
    yhat = np.asarray(search.best_estimator_.predict(X_train)).reshape(-1)
    train_r2 = r2_score(y_train, yhat) if len(y_train) >= 2 else np.nan
    return {
        "best_estimator": search.best_estimator_,
        "best_params": search.best_params_,
        "best_score": float(cvres.loc[search.best_index_, score_col]),
        "best_score_std": float(cvres.loc[search.best_index_, std_col]) if std_col in cvres.columns else 0.0,
        "train_r2": float(train_r2),
    }

def prefilter_features_groupwise(X_df, y, groups, top_k=20, corr_threshold=0.85, random_state=42):
    X = X_df.copy()
    y = np.asarray(y, dtype=float)
    grp_df = X.copy()
    grp_df["target"] = y
    grp_df["GROUP_ID"] = groups
    grp_med = grp_df.groupby("GROUP_ID").median(numeric_only=True).reset_index(drop=True)
    y_group = grp_med.pop("target").to_numpy(dtype=float)
    X_group = grp_med.copy()
    keep_cols = [c for c in X_group.columns if X_group[c].notna().sum() >= max(6, int(0.6 * len(X_group)))]
    X_group = X_group[keep_cols].copy()
    if X_group.shape[1] == 0:
        return [], pd.DataFrame(columns=["feature_name", "ensemble_score"])
    X_imp = X_group.fillna(X_group.median())
    try:
        mi = mutual_info_regression(X_imp, y_group, random_state=random_state)
    except Exception:
        mi = np.zeros(X_imp.shape[1])
    try:
        en = Pipeline([("sc", StandardScaler()), ("m", ElasticNetCV(l1_ratio=[0.2,0.5,0.8], cv=5, random_state=random_state, max_iter=10000))])
        en.fit(X_imp, y_group)
        coefs = np.abs(en.named_steps["m"].coef_)
    except Exception:
        coefs = np.zeros(X_imp.shape[1])
    score_df = pd.DataFrame({
        "feature_name": X_imp.columns,
        "spearman": [abs(spearman_corr_safe(X_imp[c].to_numpy(dtype=float), y_group)) for c in X_imp.columns],
        "pearson": [abs(pearson_corr_safe(X_imp[c].to_numpy(dtype=float), y_group)) for c in X_imp.columns],
        "mutual_info": list(mi),
        "elastic_net": list(coefs),
    })
    score_df["ensemble_score"] = (
        0.35 * minmax_series(score_df["spearman"]) +
        0.15 * minmax_series(score_df["pearson"]) +
        0.15 * minmax_series(score_df["mutual_info"]) +
        0.35 * minmax_series(score_df["elastic_net"])
    )
    score_df = score_df.sort_values("ensemble_score", ascending=False).reset_index(drop=True)
    ranked = score_df["feature_name"].tolist()
    kept = []
    for feat in ranked:
        if len(kept) >= top_k:
            break
        ok = True
        for prev in kept:
            corr = abs(pearson_corr_safe(X_imp[feat].to_numpy(dtype=float), X_imp[prev].to_numpy(dtype=float)))
            if corr >= corr_threshold:
                ok = False
                break
        if ok:
            kept.append(feat)
    return kept, score_df

def get_topk_candidates_for_output(output_name):
    fam = classify_output_family(output_name)
    return TOPK_BY_FAMILY.get(fam, FINAL_TOPK_CANDIDATES)

def get_inner_scoring_for_output(output_name):
    fam = classify_output_family(output_name)
    return INNER_SCORING_BY_FAMILY.get(fam, "r2")

def get_candidate_models_for_output(output_name):
    fam = classify_output_family(output_name)
    if fam == "vibrational":
        return ["PCR_Ridge", "PLS_Regression", "Ridge", "Bayesian_Ridge", "Huber_Regressor", "Kernel_Ridge_RBF"]
    if fam == "thermal":
        return ["PCR_Ridge", "PLS_Regression", "Ridge", "Bayesian_Ridge", "Huber_Regressor", "ElasticNet_CV", "Kernel_Ridge_RBF"]
    return ["PCR_Ridge", "PLS_Regression", "Ridge", "Bayesian_Ridge", "Huber_Regressor"]

def get_model_aware_oof_predictions(X_raw, y_raw, groups_raw, rough_cols, scoring_models=None, random_state=42):
    if scoring_models is None:
        scoring_models = MODEL_AWARE_SCORING_MODELS
    X_raw = X_raw.reset_index(drop=True)
    y_raw = pd.Series(np.asarray(y_raw, dtype=float), name="target").reset_index(drop=True)
    groups_raw = pd.Series(groups_raw).reset_index(drop=True)
    pred_lists = defaultdict(list)
    splits = list(repeated_group_splits(groups_raw.to_numpy(), n_splits=min(5, len(pd.unique(groups_raw))), n_repeats=8, random_state=random_state))
    for rep, (tr_idx, va_idx) in splits:
        X_tr_raw = X_raw.iloc[tr_idx].reset_index(drop=True)
        y_tr_raw = y_raw.iloc[tr_idx].reset_index(drop=True)
        g_tr_raw = groups_raw.iloc[tr_idx].reset_index(drop=True)
        X_va_raw = X_raw.iloc[va_idx].reset_index(drop=True)
        X_tr_g, y_tr_g, g_tr_g = aggregate_by_group_firstX(X_tr_raw[rough_cols], y_tr_raw.to_numpy(), g_tr_raw.to_numpy(), y_method="median")
        X_va = X_va_raw[rough_cols].copy()
        for model_name in scoring_models:
            try:
                fit_info = fit_search_model(X_tr_g, y_tr_g.to_numpy(), g_tr_g.to_numpy(), model_name=model_name, target_transform="raw", random_state=random_state + rep)
                pred_va = np.asarray(fit_info["best_estimator"].predict(X_va)).reshape(-1)
                for idx_global, pred in zip(va_idx, pred_va):
                    pred_lists[int(idx_global)].append(float(pred))
            except Exception:
                continue
    fallback = float(np.nanmedian(y_raw))
    return pd.Series([float(np.mean(pred_lists.get(i, [fallback]))) for i in range(len(y_raw))], name="model_aware_oof_pred")

def select_best_y_within_group(X_raw, y_raw, groups_raw, xkey_raw, rough_cols, strategy="hard_closest_oof", random_state=42):
    X_raw = X_raw.reset_index(drop=True)
    y_raw = pd.Series(np.asarray(y_raw, dtype=float), name="target").reset_index(drop=True)
    groups_raw = pd.Series(groups_raw, name="GROUP_ID").astype(str).reset_index(drop=True)
    xkey_raw = pd.Series(xkey_raw, name="XKEY").astype(str).reset_index(drop=True)
    tmp = pd.concat([X_raw.copy(), y_raw, groups_raw, xkey_raw], axis=1)
    if strategy == "singleton_raw":
        tmp["selected_as_final_y"] = 1
        tmp["selected_strategy"] = strategy
        tmp["group_median_y"] = tmp.groupby("GROUP_ID")["target"].transform("median")
        tmp["abs_to_group_median"] = (tmp["target"] - tmp["group_median_y"]).abs()
        tmp["model_aware_oof_pred"] = np.nan
        tmp["abs_resid_to_oof"] = np.nan
        selected_df = tmp.groupby("GROUP_ID", sort=False).head(1).reset_index(drop=True)
        return selected_df, tmp
    if strategy == "hard_closest_oof":
        oof_pred = get_model_aware_oof_predictions(X_raw, y_raw, groups_raw, rough_cols=rough_cols, random_state=random_state)
        tmp = pd.concat([tmp, oof_pred], axis=1)
        tmp["abs_resid_to_oof"] = (tmp["target"] - tmp["model_aware_oof_pred"]).abs()
        tmp["group_median_y"] = tmp.groupby("GROUP_ID")["target"].transform("median")
        tmp["abs_to_group_median"] = (tmp["target"] - tmp["group_median_y"]).abs()
        tmp = tmp.sort_values(["GROUP_ID", "abs_resid_to_oof", "abs_to_group_median"]).copy()
        chosen = tmp.groupby("GROUP_ID", sort=False).head(1).copy()
        chosen["selected_as_final_y"] = 1
        chosen["selected_strategy"] = strategy
        tmp["selected_as_final_y"] = 0
        tmp.loc[chosen.index, "selected_as_final_y"] = 1
        tmp["selected_strategy"] = strategy
        return chosen.reset_index(drop=True), tmp.reset_index(drop=True)
    # robust representative
    selected_rows = []
    candidate_rows = []
    for g, block in tmp.groupby("GROUP_ID", sort=False):
        rep_y = robust_group_target(block["target"].to_numpy(), z=ROBUST_TRIM_Z)
        chosen = block.iloc[[0]].copy()
        chosen["target"] = rep_y
        chosen["selected_as_final_y"] = 1
        chosen["selected_strategy"] = strategy
        chosen["group_median_y"] = float(pd.Series(block["target"]).median())
        chosen["abs_to_group_median"] = abs(rep_y - chosen["group_median_y"].iloc[0])
        chosen["model_aware_oof_pred"] = np.nan
        chosen["abs_resid_to_oof"] = np.nan
        selected_rows.append(chosen)
        b = block.copy()
        b["group_representative_y"] = rep_y
        b["selected_as_final_y"] = 0
        b["selected_strategy"] = strategy
        b["group_median_y"] = float(pd.Series(block["target"]).median())
        b["abs_to_group_median"] = (b["target"] - b["group_median_y"]).abs()
        b["model_aware_oof_pred"] = np.nan
        b["abs_resid_to_oof"] = np.nan
        candidate_rows.append(b)
    return pd.concat(selected_rows, axis=0).reset_index(drop=True), pd.concat(candidate_rows, axis=0).reset_index(drop=True)

def aggregate_test_groups_by_strategy(X_raw, y_raw, groups_raw, xkey_raw, strategy="hard_closest_oof"):
    X_raw = X_raw.reset_index(drop=True)
    y_raw = pd.Series(np.asarray(y_raw, dtype=float), name="target").reset_index(drop=True)
    groups_raw = pd.Series(groups_raw, name="GROUP_ID").astype(str).reset_index(drop=True)
    xkey_raw = pd.Series(xkey_raw, name="XKEY").astype(str).reset_index(drop=True)
    tmp = pd.concat([X_raw.copy(), y_raw, groups_raw, xkey_raw], axis=1)
    rows = []
    for g, block in tmp.groupby("GROUP_ID", sort=False):
        row = block.iloc[[0]].copy()
        if strategy == "robust_trimmed_median":
            row["target"] = robust_group_target(block["target"].to_numpy(), z=ROBUST_TRIM_Z)
        elif strategy == "singleton_raw":
            row["target"] = float(block["target"].iloc[0])
        else:
            row["target"] = float(pd.Series(block["target"]).median())
        rows.append(row)
    return pd.concat(rows, axis=0).reset_index(drop=True)

def build_feature_frequency(X_df, y, groups, candidate_cols, n_repeats=24, random_state=42):
    candidate_cols = list(candidate_cols)
    freq = pd.Series(0.0, index=candidate_cols, dtype=float)
    if not candidate_cols:
        return pd.DataFrame({"feature_name": [], "selection_frequency": []})
    splits = list(repeated_group_splits(groups, n_splits=min(5, len(pd.unique(groups))), n_repeats=n_repeats, random_state=random_state))
    if len(splits) == 0:
        return pd.DataFrame({"feature_name": candidate_cols, "selection_frequency": np.zeros(len(candidate_cols))})
    for rep, (tr_idx, va_idx) in splits:
        X_tr = X_df.iloc[tr_idx][candidate_cols].copy()
        y_tr = np.asarray(y)[tr_idx]
        if len(np.unique(y_tr)) < 2:
            continue
        try:
            pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", ElasticNetCV(l1_ratio=[0.1,0.3,0.5,0.7,0.9], alphas=np.logspace(-4,1,40), cv=min(5, len(y_tr)), random_state=random_state+rep, max_iter=30000))])
            pipe.fit(X_tr, y_tr)
            coef = np.abs(pipe.named_steps["model"].coef_)
            for f, c in zip(candidate_cols, coef):
                if abs(c) > 1e-12:
                    freq[f] += 1.0
        except Exception:
            continue
    if freq.max() > 0:
        freq = freq / freq.max()
    out = freq.sort_values(ascending=False).rename("selection_frequency").reset_index()
    out.columns = ["feature_name", "selection_frequency"]
    return out

def corr_prune_from_ranked(X_df, ranked_features, keep_k=20, threshold=0.90):
    ranked_features = [f for f in ranked_features if f in X_df.columns]
    if len(ranked_features) <= 1:
        return ranked_features[:keep_k]
    corr = X_df[ranked_features].corr(method="spearman").abs().fillna(0.0)
    selected = []
    for feat in ranked_features:
        if all(corr.loc[feat, chosen] < threshold for chosen in selected):
            selected.append(feat)
        if len(selected) >= keep_k:
            break
    return selected

def choose_best_model_and_topk(X_train, y_train, g_train, ranked_features, output_name="", target_transform_candidates=None, model_names=None, random_state=42):
    if target_transform_candidates is None:
        target_transform_candidates = TARGET_TRANSFORM_CANDIDATES
    if model_names is None:
        model_names = FINAL_MODEL_NAMES
    rows = []
    fam = classify_output_family(output_name)
    for topk in get_topk_candidates_for_output(output_name):
        feat_list = ranked_features[:min(topk, len(ranked_features))]
        if len(feat_list) < 2:
            continue
        X_sel = X_train[feat_list].copy()
        for tt in target_transform_candidates:
            for model_name in model_names:
                try:
                    fit_info = fit_search_model(X_sel, y_train, g_train, model_name=model_name, target_transform=tt, scoring=get_inner_scoring_for_output(output_name), random_state=random_state)
                    fam_topk_pen = TOPK_PENALTY * (1.25 if fam == "mechanical_energy" else 1.0)
                    stability_objective = (
                        float(fit_info["best_score"]) -
                        SUMMARY_STD_PENALTY * float(fit_info.get("best_score_std", 0.0)) -
                        fam_topk_pen * float(topk) -
                        COMPLEXITY_PENALTY * float(MODEL_COMPLEXITY_RANK.get(model_name, 10))
                    )
                    rows.append({
                        "model_name": model_name,
                        "target_transform": tt,
                        "topk": int(topk),
                        "selected_features": feat_list,
                        "best_estimator": fit_info["best_estimator"],
                        "best_params": fit_info["best_params"],
                        "search_score": float(fit_info["best_score"]),
                        "search_score_std": float(fit_info.get("best_score_std", 0.0)),
                        "stability_objective": float(stability_objective),
                        "train_r2": float(fit_info.get("train_r2", np.nan)),
                        "complexity_rank": int(MODEL_COMPLEXITY_RANK.get(model_name, 10)),
                    })
                except Exception:
                    continue
    df = pd.DataFrame(rows)
    if df.empty:
        return None, df
    best = df.sort_values(["stability_objective", "search_score", "topk", "complexity_rank"], ascending=[False, False, True, True]).iloc[0].to_dict()
    return best, df


In [7]:

# ============================================================
# Cell C2+. New helper methods: SPCA / BlockPCA / Stability / Bagged / MultiTask
# ============================================================

from sklearn.linear_model import MultiTaskElasticNetCV, LassoCV
from sklearn.base import BaseEstimator, RegressorMixin

# ----------------------------
# small utilities
# ----------------------------
def safe_power_name(name):
    return "yeo-johnson" if str(name).lower().startswith("yeo") else None

def make_y_transformer(name):
    if str(name).lower().startswith("yeo"):
        return PowerTransformer(method="yeo-johnson", standardize=True)
    return None

def build_regressor(model_name, random_state=42, n_components=None):
    if model_name == "Ridge":
        return Ridge(alpha=1.0, random_state=random_state)
    if model_name == "Huber":
        return HuberRegressor(epsilon=1.35, alpha=0.0001)
    if model_name == "PLS":
        return PLSRegression(n_components=int(max(1, n_components or 2)), scale=False)
    raise ValueError(model_name)

def fit_predict_single_model(X_train, y_train, X_test, model_name="Ridge", y_transform="raw", random_state=42, n_components=None):
    steps = [("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]
    reg = build_regressor(model_name, random_state=random_state, n_components=n_components)
    pipe = Pipeline(steps + [("model", reg)])
    ytfm = make_y_transformer(y_transform)
    if ytfm is not None:
        est = TransformedTargetRegressor(regressor=pipe, transformer=ytfm)
    else:
        est = pipe
    est.fit(X_train, y_train)
    pred = np.asarray(est.predict(X_test)).reshape(-1)
    return est, pred

def score_prediction(y_true, y_pred):
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    if mask.sum() < MIN_VALID_EVAL_SAMPLES:
        return np.nan, np.nan, np.nan
    yt = np.asarray(y_true)[mask]
    yp = np.asarray(y_pred)[mask]
    return (
        float(r2_score(yt, yp)),
        float(math.sqrt(mean_squared_error(yt, yp))),
        float(mean_absolute_error(yt, yp))
    )

def inner_group_cv_score(X, y, groups, model_name="Ridge", y_transform="raw", n_components=None,
                         n_splits=5, n_repeats=1, random_state=42):
    rows = []
    split_iter = repeated_group_splits(
        groups,
        n_splits=min(n_splits, len(pd.unique(groups))),
        n_repeats=n_repeats,
        random_state=random_state,
        return_split_id=True,
        test_size=INNER_GSS_TEST_SIZE
    )
    for cv_run_id, repeat_id, fold_id, tr_idx, te_idx in split_iter:
        Xtr = X.iloc[tr_idx].copy()
        Xte = X.iloc[te_idx].copy()
        ytr = np.asarray(y)[tr_idx]
        yte = np.asarray(y)[te_idx]
        # drop invalid rows
        train_mask = np.isfinite(ytr)
        test_mask = np.isfinite(yte)
        if train_mask.sum() < MIN_VALID_EVAL_SAMPLES or test_mask.sum() < MIN_VALID_EVAL_SAMPLES:
            continue
        Xtr = Xtr.loc[train_mask].reset_index(drop=True)
        Xte = Xte.loc[test_mask].reset_index(drop=True)
        ytr = ytr[train_mask]
        yte = yte[test_mask]
        try:
            est, pred = fit_predict_single_model(Xtr, ytr, Xte, model_name=model_name, y_transform=y_transform,
                                                 random_state=random_state + cv_run_id, n_components=n_components)
            r2, rmse, mae = score_prediction(yte, pred)
            if np.isfinite(r2):
                rows.append({"r2": r2, "rmse": rmse, "mae": mae})
        except Exception:
            continue
    if not rows:
        return {"mean_r2": np.nan, "std_r2": np.nan, "mean_rmse": np.nan, "mean_mae": np.nan}
    df = pd.DataFrame(rows)
    return {
        "mean_r2": float(df["r2"].mean()),
        "std_r2": float(0.0 if len(df) == 1 else df["r2"].std(ddof=1)),
        "mean_rmse": float(df["rmse"].mean()),
        "mean_mae": float(df["mae"].mean()),
    }

def get_prefilter_topk_for_output_local(output_name):
    fam = classify_output_family(output_name)
    return int(ROUGH_PREFILTER_TOPK_BY_FAMILY.get(fam, ROUGH_PREFILTER_TOPK))

# ----------------------------
# feature set builders
# ----------------------------
def build_ranked_features_train(X_train_sel, y_train_sel, g_train_sel, output_name, cv_run_id):
    fam = classify_output_family(output_name)
    prefilter_topk = get_prefilter_topk_for_output_local(output_name)

    final_prefilter_cols, final_score_df = prefilter_features_groupwise(
        X_train_sel, y_train_sel, g_train_sel,
        top_k=prefilter_topk,
        corr_threshold=CORR_PRUNE_THRESHOLD,
        random_state=RANDOM_STATE + 1000 + cv_run_id
    )
    if len(final_prefilter_cols) < 2:
        return [], pd.DataFrame()

    freq_df = build_feature_frequency(
        X_train_sel, y_train_sel, g_train_sel,
        candidate_cols=final_prefilter_cols,
        n_repeats=24,
        random_state=RANDOM_STATE + 2000 + cv_run_id
    )
    rank_df = final_score_df[["feature_name", "ensemble_score"]].merge(freq_df, on="feature_name", how="left").fillna(0.0)
    rank_df["rank_score"] = 0.55 * minmax_series(rank_df["selection_frequency"]) + 0.45 * minmax_series(rank_df["ensemble_score"])
    rank_df = rank_df.sort_values("rank_score", ascending=False).reset_index(drop=True)
    ranked = rank_df["feature_name"].tolist()
    ranked = corr_prune_from_ranked(
        X_train_sel[ranked], ranked,
        keep_k=max(12, MAX_FINAL_FEATURES),
        threshold=CORR_PRUNE_THRESHOLD
    )
    return ranked, rank_df

def supervised_screen_features(X_train, y_train, max_keep=16):
    cols = list(X_train.columns)
    rows = []
    for c in cols:
        x = pd.to_numeric(X_train[c], errors="coerce")
        mask = np.isfinite(x) & np.isfinite(y_train)
        if mask.sum() < 5:
            continue
        try:
            pr = np.corrcoef(x[mask], y_train[mask])[0,1]
        except Exception:
            pr = np.nan
        try:
            sp = spearmanr(x[mask], y_train[mask]).correlation
        except Exception:
            sp = np.nan
        rows.append((c, abs(0.55*(0 if pd.isna(pr) else pr) + 0.45*(0 if pd.isna(sp) else sp))))
    if not rows:
        return []
    sdf = pd.DataFrame(rows, columns=["feature","score"]).sort_values("score", ascending=False)
    return sdf["feature"].head(max_keep).tolist()

def fit_spca_transform(X_train, y_train, screen_keep=16, n_components=2):
    screen_feats = supervised_screen_features(X_train, y_train, max_keep=screen_keep)
    if len(screen_feats) < 2:
        return None
    prep = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
    Xp = prep.fit_transform(X_train[screen_feats])
    pca = PCA(n_components=min(n_components, Xp.shape[1], max(1, Xp.shape[0]-1)), random_state=RANDOM_STATE)
    Z = pca.fit_transform(Xp)
    return {"prep": prep, "pca": pca, "features": screen_feats, "Z_train": Z}

def transform_spca(obj, X):
    Xp = obj["prep"].transform(X[obj["features"]])
    return pd.DataFrame(obj["pca"].transform(Xp), index=X.index, columns=[f"SPC{i+1}" for i in range(obj["pca"].n_components_)])

def build_corr_blocks(X_df, threshold=0.65):
    corr = X_df.corr().abs().fillna(0.0)
    cols = list(corr.columns)
    unassigned = set(cols)
    blocks = []
    while unassigned:
        c = next(iter(unassigned))
        grp = [k for k in cols if (corr.loc[c, k] >= threshold)]
        grp = [g for g in grp if g in unassigned]
        if len(grp) == 0:
            grp = [c]
        blocks.append(sorted(grp))
        for g in grp:
            if g in unassigned:
                unassigned.remove(g)
    return blocks

def fit_block_pca_transform(X_train, ranked_features, corr_threshold=0.65, max_blocks=8):
    use_feats = ranked_features[:min(len(ranked_features), 20)]
    if len(use_feats) < 2:
        return None
    X_use = X_train[use_feats].copy()
    prep = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
    Xp = pd.DataFrame(prep.fit_transform(X_use), columns=use_feats, index=X_use.index)
    blocks = build_corr_blocks(Xp, threshold=corr_threshold)[:max_blocks]
    models = []
    Z_list = []
    names = []
    for bi, block in enumerate(blocks):
        pca = PCA(n_components=1, random_state=RANDOM_STATE)
        z = pca.fit_transform(Xp[block])
        models.append((block, pca))
        Z_list.append(z.reshape(-1,1))
        names.append(f"BC{bi+1}")
    Z = np.hstack(Z_list) if Z_list else np.empty((len(X_train),0))
    return {"prep": prep, "models": models, "feature_names": names, "Z_train": Z}

def transform_block_pca(obj, X):
    cols = obj["prep"].feature_names_in_
    Xp = pd.DataFrame(obj["prep"].transform(X[list(cols)]), columns=list(cols), index=X.index)
    arrs = []
    for block, pca in obj["models"]:
        arrs.append(pca.transform(Xp[block]).reshape(-1,1))
    Z = np.hstack(arrs) if arrs else np.empty((len(X),0))
    return pd.DataFrame(Z, index=X.index, columns=obj["feature_names"])

def stability_select_features(X_train, y_train, groups, ranked_features, top_keep=6, n_boot=60, subsample=0.80, random_state=42):
    feats = ranked_features[:min(len(ranked_features), 18)]
    if len(feats) < 2:
        return []
    rng = np.random.RandomState(random_state)
    uniq = np.array(pd.unique(groups))
    counts = Counter()
    for b in range(n_boot):
        n_take = max(4, int(len(uniq) * subsample))
        sel_groups = rng.choice(uniq, size=n_take, replace=False)
        mask = pd.Series(groups).isin(sel_groups).to_numpy()
        Xb = X_train.loc[mask, feats].copy()
        yb = np.asarray(y_train)[mask]
        valid = np.isfinite(yb)
        if valid.sum() < 8:
            continue
        Xb = Xb.loc[valid]
        yb = yb[valid]
        try:
            pipe = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("lasso", LassoCV(cv=4, random_state=random_state + b, n_alphas=40, max_iter=10000))
            ])
            pipe.fit(Xb, yb)
            coef = np.asarray(pipe.named_steps["lasso"].coef_).reshape(-1)
            for f, c in zip(feats, coef):
                if abs(c) > 1e-10:
                    counts[f] += 1
        except Exception:
            continue
    if not counts:
        return feats[:min(top_keep, len(feats))]
    freq = pd.DataFrame({"feature": feats, "count":[counts.get(f,0) for f in feats]})
    freq["prob"] = freq["count"] / max(1, n_boot)
    freq = freq.sort_values(["prob"], ascending=False)
    out = freq.loc[freq["prob"] >= 0.35, "feature"].tolist()
    if len(out) < 2:
        out = freq["feature"].head(min(top_keep, len(freq))).tolist()
    return out[:min(top_keep, len(out))]

class BaggedSubspaceRidge(BaseEstimator, RegressorMixin):
    def __init__(self, n_estimators=40, feature_frac=0.7, alpha=1.0, random_state=42):
        self.n_estimators = n_estimators
        self.feature_frac = feature_frac
        self.alpha = alpha
        self.random_state = random_state

    def fit(self, X, y):
        X = pd.DataFrame(X).copy()
        self.columns_ = list(X.columns)
        rng = np.random.RandomState(self.random_state)
        self.models_ = []
        min_k = max(2, int(len(self.columns_) * self.feature_frac))
        for i in range(self.n_estimators):
            feat_idx = rng.choice(len(self.columns_), size=min_k, replace=False)
            feats = [self.columns_[j] for j in feat_idx]
            pipe = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("ridge", Ridge(alpha=self.alpha, random_state=self.random_state + i))
            ])
            pipe.fit(X[feats], y)
            self.models_.append((feats, pipe))
        return self

    def predict(self, X):
        X = pd.DataFrame(X).copy()
        preds = []
        for feats, pipe in self.models_:
            preds.append(np.asarray(pipe.predict(X[feats])).reshape(-1))
        return np.mean(np.vstack(preds), axis=0)

def fit_predict_bagged_ridge(X_train, y_train, X_test, y_transform="raw", random_state=42):
    ytfm = make_y_transformer(y_transform)
    model = BaggedSubspaceRidge(n_estimators=40, feature_frac=0.7, alpha=1.0, random_state=random_state)
    if ytfm is not None:
        est = TransformedTargetRegressor(regressor=model, transformer=ytfm)
    else:
        est = model
    est.fit(X_train, y_train)
    pred = np.asarray(est.predict(X_test)).reshape(-1)
    return est, pred


# ----------------------------
# family multi-task screen
# ----------------------------
def build_family_multitask_feature_map():
    fam_map = {}
    candidate_outputs = list(DATA_BY_OUTPUT.keys())
    families = {"thermal": [], "vibrational": []}

    for out in candidate_outputs:
        fam = classify_output_family(out)
        if fam in families:
            families[fam].append(out)

    for fam, outs in families.items():
        if len(outs) < 2:
            continue

        tmp_frames = []

        for out in outs:
            bundle = DATA_BY_OUTPUT[out]
            d = bundle["df"].copy()

            xkey_col = bundle.get("xkey_col", "XKEY")
            group_col = bundle.get("group_col", "GROUP_ID")
            target_col = bundle["target_col"]

            missing_cols = [c for c in [xkey_col, group_col, target_col] if c not in d.columns]
            if missing_cols:
                print(f"[build_family_multitask_feature_map] skip {out} | missing columns: {missing_cols}")
                continue

            tmp = d[[xkey_col, group_col, target_col]].copy()
            tmp.columns = ["XKEY", "GROUP_ID", out]

            tmp["XKEY"] = tmp["XKEY"].astype(str)
            tmp["GROUP_ID"] = tmp["GROUP_ID"].astype(str)
            tmp[out] = pd.to_numeric(tmp[out], errors="coerce")

            tmp_frames.append(tmp)

        if len(tmp_frames) < 2:
            continue

        merged = tmp_frames[0]
        for t in tmp_frames[1:]:
            merged = merged.merge(t, on=["XKEY", "GROUP_ID"], how="inner")

        if not merged.empty:
            fam_map[fam] = merged.reset_index(drop=True)

    return fam_map

FAMILY_MT_DATA = build_family_multitask_feature_map()

def multitask_screen_features_for_family(output_name, X_train_sel, selected_train_df, top_keep=10, random_state=42):
    fam = classify_output_family(output_name)
    if fam not in FAMILY_MT_DATA:
        return []

    fam_df = FAMILY_MT_DATA[fam].copy()
    if fam_df.empty:
        return []

    if "GROUP_ID" not in selected_train_df.columns:
        return []

    tr_groups = set(selected_train_df["GROUP_ID"].astype(str).unique())
    fam_df["GROUP_ID"] = fam_df["GROUP_ID"].astype(str)
    fam_df = fam_df[fam_df["GROUP_ID"].isin(tr_groups)].copy()
    if fam_df.empty:
        return []

    out_cols = [c for c in fam_df.columns if c not in ("XKEY", "GROUP_ID")]
    if len(out_cols) < 2:
        return []

    if "XKEY" not in selected_train_df.columns:
        bundle = DATA_BY_OUTPUT[output_name]
        xkey_col = bundle.get("xkey_col", "XKEY")
        if xkey_col in selected_train_df.columns:
            base = selected_train_df.copy().rename(columns={xkey_col: "XKEY"})
        else:
            return []
    else:
        base = selected_train_df.copy()

    feature_cols = list(X_train_sel.columns)
    keep_cols = ["XKEY"] + [c for c in feature_cols if c in base.columns]
    base = base[keep_cols].drop_duplicates(subset=["XKEY"]).copy()
    base["XKEY"] = base["XKEY"].astype(str)

    merged = fam_df.merge(base, on="XKEY", how="inner")
    if merged.shape[0] < 12:
        return []

    Y = merged[out_cols].apply(pd.to_numeric, errors="coerce")
    valid_y = Y.notna().all(axis=1)
    merged = merged.loc[valid_y].reset_index(drop=True)
    if merged.shape[0] < 12:
        return []

    Y = merged[out_cols].to_numpy(dtype=float)
    X = merged[feature_cols].copy()

    valid_x = ~X.isna().all(axis=1)
    X = X.loc[valid_x].reset_index(drop=True)
    Y = Y[valid_x.to_numpy()]

    if X.shape[0] < 12:
        return []

    try:
        pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("mt", MultiTaskElasticNetCV(
                l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9],
                alphas=None,
                n_alphas=40,
                cv=min(5, max(2, X.shape[0] // 4)),
                random_state=random_state,
                max_iter=15000
            ))
        ])
        pipe.fit(X, Y)

        coef = np.asarray(pipe.named_steps["mt"].coef_)
        score = np.sqrt((coef ** 2).sum(axis=0))

        feat_df = pd.DataFrame({
            "feature": list(X.columns),
            "score": score
        }).sort_values("score", ascending=False)

        return feat_df["feature"].head(min(top_keep, len(feat_df))).tolist()

    except Exception as e:
        print(f"[multitask_screen_features_for_family] {output_name} failed: {type(e).__name__}: {e}")
        return []

# ----------------------------
# Method candidates
# ----------------------------
def get_method_candidates_for_output(output_name):
    fam = classify_output_family(output_name)
    methods = [
        "baseline_stability",
        "stability_lasso_ridge",
        "spca_ridge",
        "spca_huber",
        "block_pca_ridge",
        "bagged_subspace_ridge",
        "minimal_class_average",
    ]
    if fam in ("thermal", "vibrational"):
        methods += ["multitask_screen_ridge", "multitask_screen_pls"]
    if fam == "mechanical_energy":
        methods += ["spca_pls"]
    return methods

def choose_within_tolerance(rows_df, tol_frac=0.03):
    if rows_df.empty:
        return None
    best = rows_df["score_obj"].max()
    keep = rows_df[rows_df["score_obj"] >= best - abs(best) * tol_frac].copy()
    keep = keep.sort_values(["score_obj", "n_features", "complexity_rank"], ascending=[False, True, True]).reset_index(drop=True)
    return keep.iloc[0].to_dict()

def evaluate_method_train_test(method_name, output_name, X_train_sel, y_train_sel, g_train_sel, selected_train_df,
                               X_test_group, y_test_group, ranked_features, random_state=42):
    rows = []
    fam = classify_output_family(output_name)

    if method_name == "baseline_stability":
        best_choice, model_search_df = choose_best_model_and_topk(
            X_train=X_train_sel[ranked_features],
            y_train=y_train_sel,
            g_train=g_train_sel,
            ranked_features=ranked_features,
            output_name=output_name,
            target_transform_candidates=TARGET_TRANSFORM_CANDIDATES,
            model_names=get_candidate_models_for_output(output_name),
            random_state=random_state
        )
        if best_choice is None:
            return None
        feats = list(best_choice["selected_features"])
        Xtr = X_train_sel[feats].copy()
        Xte = X_test_group[feats].copy()
        pred = np.asarray(best_choice["best_estimator"].predict(Xte)).reshape(-1)
        r2, rmse, mae = score_prediction(y_test_group, pred)
        if not np.isfinite(r2):
            return None
        return {
            "method_name": method_name,
            "model_name": best_choice["model_name"],
            "target_transform": best_choice["target_transform"],
            "topk": int(best_choice["topk"]),
            "selected_features": feats,
            "n_features": len(feats),
            "complexity_rank": int(MODEL_COMPLEXITY_RANK.get(best_choice["model_name"], 10)),
            "search_score": float(best_choice["search_score"]),
            "outer_r2": r2, "outer_rmse": rmse, "outer_mae": mae,
            "fitted_estimator": best_choice["best_estimator"],
        }

    if method_name == "stability_lasso_ridge":
        feats = stability_select_features(X_train_sel, y_train_sel, g_train_sel, ranked_features, top_keep=6, random_state=random_state)
        if len(feats) < 2:
            return None
        best = None
        for tt in TARGET_TRANSFORM_CANDIDATES:
            inner = inner_group_cv_score(X_train_sel[feats], y_train_sel, g_train_sel, model_name="Ridge", y_transform=tt,
                                         n_splits=5, n_repeats=1, random_state=random_state)
            score_obj = inner["mean_r2"] - 0.10*(0 if pd.isna(inner["std_r2"]) else inner["std_r2"]) - 0.01*len(feats)
            row = {"tt": tt, "inner": inner, "score_obj": score_obj}
            if (best is None) or (score_obj > best["score_obj"]):
                best = row
        est, pred = fit_predict_single_model(X_train_sel[feats], y_train_sel, X_test_group[feats], model_name="Ridge",
                                             y_transform=best["tt"], random_state=random_state)
        r2, rmse, mae = score_prediction(y_test_group, pred)
        if not np.isfinite(r2):
            return None
        return {"method_name": method_name, "model_name":"Ridge", "target_transform":best["tt"], "topk":len(feats),
                "selected_features": feats, "n_features":len(feats), "complexity_rank":2, "search_score":best["inner"]["mean_r2"],
                "outer_r2":r2, "outer_rmse":rmse, "outer_mae":mae, "fitted_estimator":est}

    if method_name in ("spca_ridge","spca_huber","spca_pls"):
        base_model = {"spca_ridge":"Ridge", "spca_huber":"Huber", "spca_pls":"PLS"}[method_name]
        cands = []
        for screen_keep in [8, 12, 16]:
            for nc in [1, 2, 3]:
                obj = fit_spca_transform(X_train_sel[ranked_features], y_train_sel, screen_keep=screen_keep, n_components=nc)
                if obj is None:
                    continue
                Ztr = transform_spca(obj, X_train_sel[ranked_features])
                Zte = transform_spca(obj, X_test_group[ranked_features])
                for tt in TARGET_TRANSFORM_CANDIDATES:
                    inner = inner_group_cv_score(Ztr, y_train_sel, g_train_sel, model_name=base_model, y_transform=tt,
                                                 n_components=min(nc, Ztr.shape[1]), n_splits=5, n_repeats=1, random_state=random_state)
                    score_obj = inner["mean_r2"] - 0.10*(0 if pd.isna(inner["std_r2"]) else inner["std_r2"]) - 0.005*nc
                    cands.append({"obj":obj, "Ztr":Ztr, "Zte":Zte, "tt":tt, "nc":nc, "inner":inner, "score_obj":score_obj})
        if not cands:
            return None
        cdf = pd.DataFrame([{"score_obj":c["score_obj"], "nc":c["nc"], "std":c["inner"]["std_r2"]} for c in cands])
        best = cands[int(cdf["score_obj"].idxmax())]
        est, pred = fit_predict_single_model(best["Ztr"], y_train_sel, best["Zte"], model_name=base_model,
                                             y_transform=best["tt"], random_state=random_state, n_components=min(best["nc"], best["Ztr"].shape[1]))
        r2, rmse, mae = score_prediction(y_test_group, pred)
        if not np.isfinite(r2):
            return None
        feats = list(best["obj"]["features"])
        return {"method_name":method_name, "model_name":base_model, "target_transform":best["tt"], "topk":len(feats),
                "selected_features":feats, "n_features":len(feats), "complexity_rank":5 if base_model=="PLS" else 2,
                "search_score":best["inner"]["mean_r2"], "outer_r2":r2, "outer_rmse":rmse, "outer_mae":mae,
                "fitted_estimator": {"spca_obj": best["obj"], "reg_model": est}}

    if method_name == "block_pca_ridge":
        best = None
        for th in [0.55, 0.65, 0.75]:
            obj = fit_block_pca_transform(X_train_sel, ranked_features, corr_threshold=th, max_blocks=8)
            if obj is None or obj["Z_train"].shape[1] < 2:
                continue
            Ztr = transform_block_pca(obj, X_train_sel)
            Zte = transform_block_pca(obj, X_test_group)
            for tt in TARGET_TRANSFORM_CANDIDATES:
                inner = inner_group_cv_score(Ztr, y_train_sel, g_train_sel, model_name="Ridge", y_transform=tt,
                                             n_splits=5, n_repeats=1, random_state=random_state)
                score_obj = inner["mean_r2"] - 0.10*(0 if pd.isna(inner["std_r2"]) else inner["std_r2"]) - 0.005*Ztr.shape[1]
                cand = {"obj":obj, "Ztr":Ztr, "Zte":Zte, "tt":tt, "inner":inner, "score_obj":score_obj}
                if (best is None) or (score_obj > best["score_obj"]):
                    best = cand
        if best is None:
            return None
        est, pred = fit_predict_single_model(best["Ztr"], y_train_sel, best["Zte"], model_name="Ridge",
                                             y_transform=best["tt"], random_state=random_state)
        r2, rmse, mae = score_prediction(y_test_group, pred)
        if not np.isfinite(r2):
            return None
        feats = sum([blk for blk, _ in best["obj"]["models"]], [])
        return {"method_name":method_name, "model_name":"Ridge", "target_transform":best["tt"], "topk":len(feats),
                "selected_features":feats, "n_features":len(feats), "complexity_rank":3, "search_score":best["inner"]["mean_r2"],
                "outer_r2":r2, "outer_rmse":rmse, "outer_mae":mae, "fitted_estimator":{"block_obj":best["obj"], "reg_model":est}}

    if method_name == "bagged_subspace_ridge":
        cands = []
        feats = ranked_features[:min(len(ranked_features), 8)]
        if len(feats) < 2:
            return None
        for tt in TARGET_TRANSFORM_CANDIDATES:
            # simple inner scoring with repeated fits
            inner = inner_group_cv_score(X_train_sel[feats], y_train_sel, g_train_sel, model_name="Ridge", y_transform=tt,
                                         n_splits=5, n_repeats=1, random_state=random_state)
            score_obj = inner["mean_r2"] - 0.10*(0 if pd.isna(inner["std_r2"]) else inner["std_r2"]) - 0.01*len(feats)
            cands.append({"tt":tt, "inner":inner, "score_obj":score_obj})
        best = max(cands, key=lambda z: z["score_obj"])
        est, pred = fit_predict_bagged_ridge(X_train_sel[feats], y_train_sel, X_test_group[feats],
                                             y_transform=best["tt"], random_state=random_state)
        r2, rmse, mae = score_prediction(y_test_group, pred)
        if not np.isfinite(r2):
            return None
        return {"method_name":method_name, "model_name":"BaggedSubspaceRidge", "target_transform":best["tt"], "topk":len(feats),
                "selected_features":feats, "n_features":len(feats), "complexity_rank":4, "search_score":best["inner"]["mean_r2"],
                "outer_r2":r2, "outer_rmse":rmse, "outer_mae":mae, "fitted_estimator":est}

    if method_name in ("multitask_screen_ridge","multitask_screen_pls"):
        base_model = "Ridge" if method_name.endswith("ridge") else "PLS"
        mt_feats = multitask_screen_features_for_family(output_name, X_train_sel[ranked_features], selected_train_df, top_keep=8, random_state=random_state)
        if len(mt_feats) < 2:
            return None
        best = None
        for topk in [2,3,4,6]:
            feats = mt_feats[:min(topk, len(mt_feats))]
            for tt in TARGET_TRANSFORM_CANDIDATES:
                inner = inner_group_cv_score(X_train_sel[feats], y_train_sel, g_train_sel, model_name=base_model, y_transform=tt,
                                             n_components=min(3, len(feats)), n_splits=5, n_repeats=1, random_state=random_state)
                score_obj = inner["mean_r2"] - 0.10*(0 if pd.isna(inner["std_r2"]) else inner["std_r2"]) - 0.01*len(feats)
                cand = {"feats":feats, "tt":tt, "inner":inner, "score_obj":score_obj}
                if (best is None) or (score_obj > best["score_obj"]):
                    best = cand
        if best is None:
            return None
        est, pred = fit_predict_single_model(X_train_sel[best["feats"]], y_train_sel, X_test_group[best["feats"]],
                                             model_name=base_model, y_transform=best["tt"],
                                             random_state=random_state, n_components=min(3, len(best["feats"])))
        r2, rmse, mae = score_prediction(y_test_group, pred)
        if not np.isfinite(r2):
            return None
        return {"method_name":method_name, "model_name":base_model, "target_transform":best["tt"], "topk":len(best["feats"]),
                "selected_features":best["feats"], "n_features":len(best["feats"]), "complexity_rank":4 if base_model=="PLS" else 2,
                "search_score":best["inner"]["mean_r2"], "outer_r2":r2, "outer_rmse":rmse, "outer_mae":mae,
                "fitted_estimator":est}

    if method_name == "minimal_class_average":
        # build a small class of close-performing models and average them
        best_choice, search_df = choose_best_model_and_topk(
            X_train=X_train_sel[ranked_features],
            y_train=y_train_sel,
            g_train=g_train_sel,
            ranked_features=ranked_features,
            output_name=output_name,
            target_transform_candidates=TARGET_TRANSFORM_CANDIDATES,
            model_names=get_candidate_models_for_output(output_name),
            random_state=random_state
        )
        if best_choice is None or search_df.empty:
            return None
        sdf = search_df.copy().sort_values(["stability_objective", "search_score_std", "topk"], ascending=[False, True, True]).reset_index(drop=True)
        best_obj = float(sdf["stability_objective"].max())
        sdf = sdf[sdf["stability_objective"] >= best_obj - abs(best_obj)*0.03].head(3).copy()
        if sdf.empty:
            return None
        preds = []
        ests = []
        for _, row in sdf.iterrows():
            feats = list(row["selected_features"])
            try:
                est, pred = fit_predict_single_model(X_train_sel[feats], y_train_sel, X_test_group[feats],
                                                     model_name=row["model_name"].replace("_Regression","").replace("PCR_Ridge","Ridge") if row["model_name"] not in ("PLS_Regression","PCR_Ridge") else ("PLS" if row["model_name"]=="PLS_Regression" else "Ridge"),
                                                     y_transform=row["target_transform"], random_state=random_state,
                                                     n_components=min(3, len(feats)))
            except Exception:
                # fallback to existing estimator if compatible
                try:
                    pred = np.asarray(row["best_estimator"].predict(X_test_group[feats])).reshape(-1)
                    est = row["best_estimator"]
                except Exception:
                    continue
            preds.append(pred)
            ests.append({"feats":feats, "name":row["model_name"], "est":est, "w":max(1e-6, float(row["stability_objective"]))})
        if len(preds) < 2:
            return None
        W = np.array([e["w"] for e in ests], dtype=float)
        W = W / W.sum()
        pred = np.average(np.vstack(preds), axis=0, weights=W)
        r2, rmse, mae = score_prediction(y_test_group, pred)
        if not np.isfinite(r2):
            return None
        feats = sorted(set(sum([e["feats"] for e in ests], [])))
        return {"method_name":method_name, "model_name":"MinimalClassAverage", "target_transform":"mixed", "topk":len(feats),
                "selected_features":feats, "n_features":len(feats), "complexity_rank":6, "search_score":float(np.mean(W)),
                "outer_r2":r2, "outer_rmse":rmse, "outer_mae":mae, "fitted_estimator":{"members":ests, "weights":W}}

    return None



# ============================================================
# Cell C2++. Additional untried methods: ARD / OMP / Quantile / Residual-ExtraTrees
# ============================================================

from sklearn.linear_model import ARDRegression, OrthogonalMatchingPursuitCV, QuantileRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.base import BaseEstimator, RegressorMixin, clone

_BASE_GET_METHOD_CANDIDATES_FOR_OUTPUT = get_method_candidates_for_output
_BASE_EVALUATE_METHOD_TRAIN_TEST = evaluate_method_train_test

class OMPThenRidge(BaseEstimator, RegressorMixin):
    def __init__(self, max_nonzero_coefs=None, alpha=1.0):
        self.max_nonzero_coefs = max_nonzero_coefs
        self.alpha = alpha

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)
        self.selector_ = OrthogonalMatchingPursuitCV(max_iter=self.max_nonzero_coefs)
        self.selector_.fit(X, y)
        coef = np.asarray(self.selector_.coef_).reshape(-1)
        self.support_idx_ = np.where(np.abs(coef) > 1e-12)[0]
        if len(self.support_idx_) == 0:
            self.support_idx_ = np.array([int(np.argmax(np.abs(coef)))]) if coef.size else np.array([], dtype=int)
        self.reg_ = Ridge(alpha=self.alpha)
        if len(self.support_idx_) > 0:
            self.reg_.fit(X[:, self.support_idx_], y)
        else:
            self.reg_.fit(X, y)
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        if hasattr(self, "support_idx_") and len(self.support_idx_) > 0:
            return self.reg_.predict(X[:, self.support_idx_])
        return self.reg_.predict(X)


class ResidualExtraTreesRegressor(BaseEstimator, RegressorMixin):
    def __init__(self, base_model="ridge", n_components=2, random_state=42):
        self.base_model = base_model
        self.n_components = n_components
        self.random_state = random_state

    def _make_base(self):
        if str(self.base_model).lower().startswith("pls"):
            return PLSRegression(n_components=int(max(1, self.n_components)), scale=False)
        return Ridge(alpha=1.0, random_state=self.random_state)

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)
        self.base_ = self._make_base()
        self.base_.fit(X, y)
        base_pred = np.asarray(self.base_.predict(X)).reshape(-1)
        resid = y - base_pred
        self.resid_model_ = ExtraTreesRegressor(
            n_estimators=250,
            max_depth=3,
            min_samples_leaf=3,
            max_features="sqrt",
            random_state=self.random_state
        )
        self.resid_model_.fit(X, resid)
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        base_pred = np.asarray(self.base_.predict(X)).reshape(-1)
        resid_pred = np.asarray(self.resid_model_.predict(X)).reshape(-1)
        return base_pred + resid_pred


def fit_predict_custom_estimator(X_train, y_train, X_test, estimator_obj, y_transform="raw", random_state=42):
    steps = [("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", estimator_obj)]
    pipe = Pipeline(steps)
    ytfm = make_y_transformer(y_transform)
    if ytfm is not None:
        est = TransformedTargetRegressor(regressor=pipe, transformer=ytfm)
    else:
        est = pipe
    est.fit(X_train, y_train)
    pred = np.asarray(est.predict(X_test)).reshape(-1)
    return est, pred


def inner_group_cv_score_custom(X, y, groups, estimator_factory, y_transform="raw",
                                n_splits=5, n_repeats=1, random_state=42):
    rows = []
    split_iter = repeated_group_splits(
        groups,
        n_splits=min(n_splits, len(pd.unique(groups))),
        n_repeats=n_repeats,
        random_state=random_state,
        return_split_id=True,
        test_size=INNER_GSS_TEST_SIZE
    )
    for cv_run_id, repeat_id, fold_id, tr_idx, te_idx in split_iter:
        Xtr = X.iloc[tr_idx].copy()
        Xte = X.iloc[te_idx].copy()
        ytr = np.asarray(y)[tr_idx]
        yte = np.asarray(y)[te_idx]
        train_mask = np.isfinite(ytr)
        test_mask = np.isfinite(yte)
        if train_mask.sum() < MIN_VALID_EVAL_SAMPLES or test_mask.sum() < MIN_VALID_EVAL_SAMPLES:
            continue
        Xtr = Xtr.loc[train_mask].reset_index(drop=True)
        Xte = Xte.loc[test_mask].reset_index(drop=True)
        ytr = ytr[train_mask]
        yte = yte[test_mask]
        try:
            est, pred = fit_predict_custom_estimator(
                Xtr, ytr, Xte,
                estimator_obj=estimator_factory(random_state=random_state + cv_run_id),
                y_transform=y_transform,
                random_state=random_state + cv_run_id
            )
            r2, rmse, mae = score_prediction(yte, pred)
            if np.isfinite(r2):
                rows.append({"r2": r2, "rmse": rmse, "mae": mae})
        except Exception:
            continue
    if not rows:
        return {"mean_r2": np.nan, "std_r2": np.nan, "mean_rmse": np.nan, "mean_mae": np.nan}
    df = pd.DataFrame(rows)
    return {
        "mean_r2": float(df["r2"].mean()),
        "std_r2": float(0.0 if len(df) == 1 else df["r2"].std(ddof=1)),
        "mean_rmse": float(df["rmse"].mean()),
        "mean_mae": float(df["mae"].mean()),
    }


def _custom_method_feature_grid(output_name):
    fam = classify_output_family(output_name)
    if fam == "mechanical_energy":
        return [2, 3, 4, 6]
    return [2, 3, 4, 6, 8]


def get_method_candidates_for_output(output_name):
    methods = _BASE_GET_METHOD_CANDIDATES_FOR_OUTPUT(output_name)
    fam = classify_output_family(output_name)
    extra = [
        "direct_ard",
        "direct_elasticnet_cv",
        "direct_omp_ridge",
    ]
    if fam in ("thermal", "vibrational"):
        extra += [
            "direct_quantile_median",
            "residual_ridge_extratrees",
            "residual_pls_extratrees",
        ]
    else:
        extra += [
            "residual_ridge_extratrees",
        ]
    # deduplicate while preserving order
    seen = set()
    out = []
    for m in methods + extra:
        if m not in seen:
            out.append(m)
            seen.add(m)
    return out


def evaluate_method_train_test(method_name, output_name, X_train_sel, y_train_sel, g_train_sel, selected_train_df,
                               X_test_group, y_test_group, ranked_features, random_state=42):
    # handle new methods first
    custom_methods = {
        "direct_ard",
        "direct_elasticnet_cv",
        "direct_omp_ridge",
        "direct_quantile_median",
        "residual_ridge_extratrees",
        "residual_pls_extratrees",
    }
    if method_name not in custom_methods:
        return _BASE_EVALUATE_METHOD_TRAIN_TEST(
            method_name=method_name,
            output_name=output_name,
            X_train_sel=X_train_sel,
            y_train_sel=y_train_sel,
            g_train_sel=g_train_sel,
            selected_train_df=selected_train_df,
            X_test_group=X_test_group,
            y_test_group=y_test_group,
            ranked_features=ranked_features,
            random_state=random_state,
        )

    fam = classify_output_family(output_name)
    feature_grid = _custom_method_feature_grid(output_name)

    def factory_for(method_name, n_feats):
        if method_name == "direct_ard":
            return lambda random_state=42: ARDRegression()
        if method_name == "direct_elasticnet_cv":
            return lambda random_state=42: ElasticNetCV(
                l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9],
                cv=4,
                random_state=random_state,
                n_alphas=60,
                max_iter=15000
            )
        if method_name == "direct_omp_ridge":
            return lambda random_state=42: OMPThenRidge(max_nonzero_coefs=min(max(2, n_feats), 6), alpha=1.0)
        if method_name == "direct_quantile_median":
            return lambda random_state=42: QuantileRegressor(
                quantile=0.5,
                alpha=0.01,
                solver="highs"
            )
        if method_name == "residual_ridge_extratrees":
            return lambda random_state=42: ResidualExtraTreesRegressor(base_model="ridge", n_components=min(3, n_feats), random_state=random_state)
        if method_name == "residual_pls_extratrees":
            return lambda random_state=42: ResidualExtraTreesRegressor(base_model="pls", n_components=min(3, n_feats), random_state=random_state)
        raise ValueError(method_name)

    best = None
    target_transforms = TARGET_TRANSFORM_CANDIDATES
    # quantile regression often behaves best without transform; still compare raw/yeo
    for topk in feature_grid:
        feats = ranked_features[:min(topk, len(ranked_features))]
        if len(feats) < 2:
            continue
        Xtr = X_train_sel[feats].copy()
        Xte = X_test_group[feats].copy()

        for tt in target_transforms:
            inner = inner_group_cv_score_custom(
                Xtr, y_train_sel, g_train_sel,
                estimator_factory=factory_for(method_name, len(feats)),
                y_transform=tt,
                n_splits=5,
                n_repeats=1,
                random_state=random_state
            )
            if pd.isna(inner["mean_r2"]):
                continue
            complexity_rank = {
                "direct_ard": 2,
                "direct_elasticnet_cv": 3,
                "direct_omp_ridge": 3,
                "direct_quantile_median": 3,
                "residual_ridge_extratrees": 5,
                "residual_pls_extratrees": 6,
            }[method_name]
            score_obj = (
                inner["mean_r2"]
                - 0.12 * (0 if pd.isna(inner["std_r2"]) else inner["std_r2"])
                - 0.008 * len(feats)
                - 0.01 * complexity_rank
            )
            cand = {
                "feats": feats,
                "tt": tt,
                "inner": inner,
                "score_obj": score_obj,
                "complexity_rank": complexity_rank,
            }
            if (best is None) or (score_obj > best["score_obj"]):
                best = cand

    if best is None:
        return None

    est, pred = fit_predict_custom_estimator(
        X_train_sel[best["feats"]], y_train_sel, X_test_group[best["feats"]],
        estimator_obj=factory_for(method_name, len(best["feats"]))(random_state=random_state),
        y_transform=best["tt"],
        random_state=random_state
    )
    r2, rmse, mae = score_prediction(y_test_group, pred)
    if not np.isfinite(r2):
        return None

    model_name_map = {
        "direct_ard": "ARD_Regression",
        "direct_elasticnet_cv": "ElasticNet_CV_Direct",
        "direct_omp_ridge": "OMP_then_Ridge",
        "direct_quantile_median": "Quantile_Median",
        "residual_ridge_extratrees": "Residual_Ridge_ExtraTrees",
        "residual_pls_extratrees": "Residual_PLS_ExtraTrees",
    }

    return {
        "method_name": method_name,
        "model_name": model_name_map[method_name],
        "target_transform": best["tt"],
        "topk": len(best["feats"]),
        "selected_features": list(best["feats"]),
        "n_features": len(best["feats"]),
        "complexity_rank": best["complexity_rank"],
        "search_score": float(best["inner"]["mean_r2"]),
        "outer_r2": float(r2),
        "outer_rmse": float(rmse),
        "outer_mae": float(mae),
        "fitted_estimator": est,
    }


# ============================================================
# Cell C2++ Override: safer multitask + extra methods
# ============================================================
from sklearn.linear_model import ARDRegression, ElasticNetCV, OrthogonalMatchingPursuitCV, TheilSenRegressor, RANSACRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, RationalQuadratic, WhiteKernel, ConstantKernel
from sklearn.kernel_ridge import KernelRidge

# --- safer family multitask map (override old version) ---
def build_family_multitask_feature_map():
    fam_map = {}
    candidate_outputs = list(DATA_BY_OUTPUT.keys())
    families = {"thermal": [], "vibrational": []}
    for out in candidate_outputs:
        fam = classify_output_family(out)
        if fam in families:
            families[fam].append(out)

    for fam, outs in families.items():
        if len(outs) < 2:
            continue
        tmp_frames = []
        for out in outs:
            bundle = DATA_BY_OUTPUT[out]
            d = bundle["df"].copy()
            xkey_col = bundle.get("xkey_col", "XKEY")
            group_col = bundle.get("group_col", "GROUP_ID")
            target_col = bundle["target_col"]
            missing_cols = [c for c in [xkey_col, group_col, target_col] if c not in d.columns]
            if missing_cols:
                print(f"[build_family_multitask_feature_map] skip {out} | missing columns: {missing_cols}")
                continue
            tmp = d[[xkey_col, group_col, target_col]].copy()
            tmp.columns = ["XKEY", "GROUP_ID", out]
            tmp["XKEY"] = tmp["XKEY"].astype(str)
            tmp["GROUP_ID"] = tmp["GROUP_ID"].astype(str)
            tmp[out] = pd.to_numeric(tmp[out], errors="coerce")
            tmp_frames.append(tmp)
        if len(tmp_frames) < 2:
            continue
        merged = tmp_frames[0]
        for t in tmp_frames[1:]:
            merged = merged.merge(t, on=["XKEY", "GROUP_ID"], how="inner")
        if not merged.empty:
            fam_map[fam] = merged.reset_index(drop=True)
    return fam_map

FAMILY_MT_DATA = build_family_multitask_feature_map()

def multitask_screen_features_for_family(output_name, X_train_sel, selected_train_df, top_keep=10, random_state=42):
    fam = classify_output_family(output_name)
    if fam not in FAMILY_MT_DATA:
        return []
    fam_df = FAMILY_MT_DATA[fam].copy()
    if fam_df.empty or "GROUP_ID" not in selected_train_df.columns:
        return []
    tr_groups = set(selected_train_df["GROUP_ID"].astype(str).unique())
    fam_df["GROUP_ID"] = fam_df["GROUP_ID"].astype(str)
    fam_df = fam_df[fam_df["GROUP_ID"].isin(tr_groups)].copy()
    if fam_df.empty:
        return []
    out_cols = [c for c in fam_df.columns if c not in ("XKEY","GROUP_ID")]
    if len(out_cols) < 2:
        return []
    base = selected_train_df.copy()
    if "XKEY" not in base.columns:
        bundle = DATA_BY_OUTPUT[output_name]
        xkey_col = bundle.get("xkey_col", "XKEY")
        if xkey_col in base.columns:
            base = base.rename(columns={xkey_col: "XKEY"})
        else:
            return []
    feature_cols = list(X_train_sel.columns)
    keep_cols = ["XKEY"] + [c for c in feature_cols if c in base.columns]
    base = base[keep_cols].drop_duplicates(subset=["XKEY"]).copy()
    base["XKEY"] = base["XKEY"].astype(str)
    merged = fam_df.merge(base, on="XKEY", how="inner")
    if merged.shape[0] < 12:
        return []
    Y = merged[out_cols].apply(pd.to_numeric, errors="coerce")
    valid_y = Y.notna().all(axis=1)
    merged = merged.loc[valid_y].reset_index(drop=True)
    if merged.shape[0] < 12:
        return []
    Y = merged[out_cols].to_numpy(dtype=float)
    X = merged[feature_cols].copy()
    valid_x = ~X.isna().all(axis=1)
    X = X.loc[valid_x].reset_index(drop=True)
    Y = Y[valid_x.to_numpy()]
    if X.shape[0] < 12:
        return []
    try:
        pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("mt", MultiTaskElasticNetCV(
                l1_ratio=[0.1,0.3,0.5,0.7,0.9],
                alphas=None,
                n_alphas=40,
                cv=min(5, max(2, X.shape[0]//4)),
                random_state=random_state,
                max_iter=15000
            ))
        ])
        pipe.fit(X, Y)
        coef = np.asarray(pipe.named_steps["mt"].coef_)
        score = np.sqrt((coef**2).sum(axis=0))
        feat_df = pd.DataFrame({"feature": list(X.columns), "score": score}).sort_values("score", ascending=False)
        return feat_df["feature"].head(min(top_keep, len(feat_df))).tolist()
    except Exception as e:
        print(f"[multitask_screen_features_for_family] {output_name} failed: {type(e).__name__}: {e}")
        return []


def fit_predict_special_method(X_train, y_train, X_test, method_name, y_transform="raw", random_state=42):
    # y_transform intentionally ignored for quantile/theilsen/ransac robust cases except where practical
    if method_name == "direct_ard":
        reg = ARDRegression()
        pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", reg)])
        est = TransformedTargetRegressor(regressor=pipe, transformer=make_y_transformer(y_transform)) if make_y_transformer(y_transform) is not None else pipe
        est.fit(X_train, y_train)
        return est, np.asarray(est.predict(X_test)).reshape(-1)
    if method_name == "direct_elasticnet_cv":
        reg = ElasticNetCV(l1_ratio=[.1,.3,.5,.7,.9,.95,.99], cv=5, random_state=random_state, max_iter=20000)
        pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", reg)])
        est = TransformedTargetRegressor(regressor=pipe, transformer=make_y_transformer(y_transform)) if make_y_transformer(y_transform) is not None else pipe
        est.fit(X_train, y_train)
        return est, np.asarray(est.predict(X_test)).reshape(-1)
    if method_name == "direct_omp_ridge":
        # OMP select then ridge refit
        prep = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
        Xtr = prep.fit_transform(X_train)
        Xte = prep.transform(X_test)
        omp = OrthogonalMatchingPursuitCV(cv=5)
        omp.fit(Xtr, y_train)
        coef = np.asarray(omp.coef_).reshape(-1)
        idx = np.flatnonzero(np.abs(coef) > 1e-12)
        if len(idx) == 0:
            idx = np.arange(min(2, Xtr.shape[1]))
        ridge = Ridge(alpha=1.0, random_state=random_state)
        ridge.fit(Xtr[:, idx], y_train)
        pred = ridge.predict(Xte[:, idx])
        return {"prep":prep, "omp":omp, "ridge":ridge, "idx":idx, "columns":list(X_train.columns)}, np.asarray(pred).reshape(-1)
    if method_name == "direct_theilsen":
        reg = TheilSenRegressor(random_state=random_state, max_subpopulation=1e4)
        pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", reg)])
        pipe.fit(X_train, y_train)
        return pipe, np.asarray(pipe.predict(X_test)).reshape(-1)
    if method_name == "direct_ransac_ridge":
        base = Ridge(alpha=1.0, random_state=random_state)
        try:
            reg = RANSACRegressor(estimator=base, random_state=random_state)
        except TypeError:
            reg = RANSACRegressor(base_estimator=base, random_state=random_state)
        pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", reg)])
        pipe.fit(X_train, y_train)
        return pipe, np.asarray(pipe.predict(X_test)).reshape(-1)
    if method_name == "direct_gpr_matern":
        prep = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
        Xtr = prep.fit_transform(X_train)
        Xte = prep.transform(X_test)
        kernel = ConstantKernel(1.0, (1e-3,1e3)) * Matern(length_scale=np.ones(Xtr.shape[1]), nu=1.5) + WhiteKernel(noise_level=1e-3)
        gpr = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True, random_state=random_state, n_restarts_optimizer=1)
        gpr.fit(Xtr, y_train)
        pred = gpr.predict(Xte)
        return {"prep":prep, "gpr":gpr, "columns":list(X_train.columns)}, np.asarray(pred).reshape(-1)
    if method_name == "direct_gpr_rq":
        prep = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
        Xtr = prep.fit_transform(X_train)
        Xte = prep.transform(X_test)
        kernel = ConstantKernel(1.0, (1e-3,1e3)) * RationalQuadratic(length_scale=1.0, alpha=1.0) + WhiteKernel(noise_level=1e-3)
        gpr = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True, random_state=random_state, n_restarts_optimizer=1)
        gpr.fit(Xtr, y_train)
        pred = gpr.predict(Xte)
        return {"prep":prep, "gpr":gpr, "columns":list(X_train.columns)}, np.asarray(pred).reshape(-1)
    if method_name == "direct_quantile_median":
        reg = QuantileRegressor(quantile=0.5, alpha=0.001, solver="highs")
        pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", reg)])
        pipe.fit(X_train, y_train)
        return pipe, np.asarray(pipe.predict(X_test)).reshape(-1)
    raise ValueError(method_name)


def _model_complexity_override(name):
    mapping = {
        "ARD_Regression": 2,
        "ElasticNetCV": 2,
        "OMP_Ridge": 3,
        "TheilSen": 3,
        "RANSAC_Ridge": 3,
        "GaussianProcess_Matern": 6,
        "GaussianProcess_RQ": 6,
        "QuantileMedian": 2,
        "Residual_Huber_KRR": 5,
        "Residual_ARD_KRR": 5,
    }
    return mapping.get(name, 10)

# override method candidate function with extra methods
_old_get_method_candidates_for_output = get_method_candidates_for_output

def get_method_candidates_for_output(output_name):
    fam = classify_output_family(output_name)
    methods = _old_get_method_candidates_for_output(output_name)
    methods += [
        "direct_theilsen",
        "direct_ransac_ridge",
        "direct_gpr_matern",
        "direct_gpr_rq",
        "residual_huber_krr",
        "residual_ard_krr",
    ]
    if fam != "vibrational":
        methods += ["direct_quantile_median"]
    return list(dict.fromkeys(methods))

_old_evaluate_method_train_test = evaluate_method_train_test

def evaluate_method_train_test(method_name, output_name, X_train_sel, y_train_sel, g_train_sel, selected_train_df,
                               X_test_group, y_test_group, ranked_features, random_state=42):
    # keep old behavior first
    old_methods = {
        "baseline_stability","stability_lasso_ridge","spca_ridge","spca_huber","block_pca_ridge",
        "bagged_subspace_ridge","minimal_class_average","multitask_screen_ridge","multitask_screen_pls",
        "spca_pls","direct_ard","direct_elasticnet_cv","direct_omp_ridge","direct_quantile_median",
        "residual_ridge_extratrees","residual_pls_extratrees"
    }
    if method_name in old_methods:
        return _old_evaluate_method_train_test(method_name, output_name, X_train_sel, y_train_sel, g_train_sel,
                                               selected_train_df, X_test_group, y_test_group, ranked_features,
                                               random_state=random_state)

    # new direct robust/GP methods
    if method_name in ("direct_theilsen","direct_ransac_ridge","direct_gpr_matern","direct_gpr_rq"):
        best = None
        for topk in [2,3,4,6]:
            feats = ranked_features[:min(topk, len(ranked_features))]
            if len(feats) < 2:
                continue
            # robust methods mostly without transform except GPR optionally
            tt_list = ["raw"] if method_name in ("direct_theilsen","direct_ransac_ridge") else TARGET_TRANSFORM_CANDIDATES
            for tt in tt_list:
                try:
                    # lightweight proxy inner score: use ridge-like scorer or direct fit in split loop once
                    inner = inner_group_cv_score(X_train_sel[feats], y_train_sel, g_train_sel,
                                                 model_name="Ridge", y_transform=tt,
                                                 n_splits=5, n_repeats=1, random_state=random_state)
                    score_obj = inner["mean_r2"] - 0.10*(0 if pd.isna(inner["std_r2"]) else inner["std_r2"]) - 0.01*len(feats)
                    cand = {"feats":feats, "tt":tt, "inner":inner, "score_obj":score_obj}
                    if (best is None) or (score_obj > best["score_obj"]):
                        best = cand
                except Exception:
                    continue
        if best is None:
            return None
        est, pred = fit_predict_special_method(X_train_sel[best["feats"]], y_train_sel, X_test_group[best["feats"]],
                                               method_name=method_name, y_transform=best["tt"], random_state=random_state)
        r2, rmse, mae = score_prediction(y_test_group, pred)
        if not np.isfinite(r2):
            return None
        model_name = {
            "direct_theilsen":"TheilSen",
            "direct_ransac_ridge":"RANSAC_Ridge",
            "direct_gpr_matern":"GaussianProcess_Matern",
            "direct_gpr_rq":"GaussianProcess_RQ",
        }[method_name]
        return {"method_name": method_name, "model_name": model_name, "target_transform": best["tt"], "topk": len(best["feats"]),
                "selected_features": best["feats"], "n_features": len(best["feats"]),
                "complexity_rank": _model_complexity_override(model_name), "search_score": best["inner"]["mean_r2"],
                "outer_r2": r2, "outer_rmse": rmse, "outer_mae": mae, "fitted_estimator": est}

    if method_name in ("residual_huber_krr", "residual_ard_krr"):
        base_model = "Huber" if method_name == "residual_huber_krr" else "Ridge"
        # ARD approximated via direct special fit for base prediction
        best = None
        for topk in [2,3,4,6]:
            feats = ranked_features[:min(topk, len(ranked_features))]
            if len(feats) < 2:
                continue
            for tt in TARGET_TRANSFORM_CANDIDATES:
                inner = inner_group_cv_score(X_train_sel[feats], y_train_sel, g_train_sel,
                                             model_name=("Huber" if method_name == "residual_huber_krr" else "Ridge"),
                                             y_transform=tt, n_splits=5, n_repeats=1, random_state=random_state)
                score_obj = inner["mean_r2"] - 0.10*(0 if pd.isna(inner["std_r2"]) else inner["std_r2"]) - 0.01*len(feats)
                cand = {"feats":feats, "tt":tt, "inner":inner, "score_obj":score_obj}
                if (best is None) or (score_obj > best["score_obj"]):
                    best = cand
        if best is None:
            return None
        feats = best["feats"]
        # fit base
        if method_name == "residual_huber_krr":
            base_est, base_pred_tr = fit_predict_single_model(X_train_sel[feats], y_train_sel, X_train_sel[feats], model_name="Huber", y_transform=best["tt"], random_state=random_state)
            base_pred_te = np.asarray(base_est.predict(X_test_group[feats])).reshape(-1)
        else:
            base_est, base_pred_tr = fit_predict_special_method(X_train_sel[feats], y_train_sel, X_train_sel[feats], method_name="direct_ard", y_transform=best["tt"], random_state=random_state)
            # predict test for ARD special object
            try:
                base_pred_te = np.asarray(base_est.predict(X_test_group[feats])).reshape(-1)
            except Exception:
                # TransformedTargetRegressor/pipeline path handled above; dict shouldn't happen for direct_ard
                return None
        resid = y_train_sel - np.asarray(base_pred_tr).reshape(-1)
        prep = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
        Xtr = prep.fit_transform(X_train_sel[feats])
        Xte = prep.transform(X_test_group[feats])
        krr = KernelRidge(alpha=1.0, kernel='rbf', gamma=None)
        krr.fit(Xtr, resid)
        pred = base_pred_te + krr.predict(Xte)
        r2, rmse, mae = score_prediction(y_test_group, pred)
        if not np.isfinite(r2):
            return None
        model_name = "Residual_Huber_KRR" if method_name == "residual_huber_krr" else "Residual_ARD_KRR"
        fitted = {"base_estimator": base_est, "krr": krr, "prep": prep, "feats": feats, "base_kind": method_name, "transform": best["tt"]}
        return {"method_name": method_name, "model_name": model_name, "target_transform": best["tt"], "topk": len(feats),
                "selected_features": feats, "n_features": len(feats), "complexity_rank": _model_complexity_override(model_name),
                "search_score": best["inner"]["mean_r2"], "outer_r2": r2, "outer_rmse": rmse, "outer_mae": mae,
                "fitted_estimator": fitted}

    return None


In [8]:

# ============================================================
# Cell D1. Method-comparison outer CV (SAFE VERSION)
# ============================================================

METHOD_SUMMARY_ROWS = []
PER_OUTPUT_RESULTS = {}

def _ensure_standard_cols(df_in, group_col=None, xkey_col=None, group_values=None, xkey_values=None):
    df = df_in.copy()
    if "GROUP_ID" not in df.columns:
        if group_col is not None and group_col in df.columns:
            df = df.rename(columns={group_col: "GROUP_ID"})
        elif group_values is not None and len(df) == len(group_values):
            df["GROUP_ID"] = group_values[:len(df)]
    if "XKEY" not in df.columns:
        if xkey_col is not None and xkey_col in df.columns:
            df = df.rename(columns={xkey_col: "XKEY"})
        elif xkey_values is not None and len(df) == len(xkey_values):
            df["XKEY"] = xkey_values[:len(df)]
    if "GROUP_ID" in df.columns:
        df["GROUP_ID"] = df["GROUP_ID"].astype(str)
    if "XKEY" in df.columns:
        df["XKEY"] = df["XKEY"].astype(str)
    if "target" in df.columns:
        df["target"] = pd.to_numeric(df["target"], errors="coerce")
    return df

for output_name, bundle in DATA_BY_OUTPUT.items():
    df = bundle["df"].copy()
    feature_cols = bundle["feature_cols"]
    target_col = bundle["target_col"]
    group_col = bundle["group_col"]
    xkey_col = bundle["xkey_col"]

    X_raw_full = df[feature_cols].copy()
    y_raw_full = pd.to_numeric(df[target_col], errors="coerce").to_numpy(dtype=float)
    g_raw_full = df[group_col].astype(str).to_numpy()
    xkey_full = df[xkey_col].astype(str).to_numpy()

    base_mask = np.isfinite(y_raw_full) & pd.notna(g_raw_full) & pd.notna(xkey_full)
    X_raw_full = X_raw_full.loc[base_mask].reset_index(drop=True)
    y_raw_full = y_raw_full[base_mask]
    g_raw_full = g_raw_full[base_mask]
    xkey_full = xkey_full[base_mask]

    n_groups = len(pd.unique(g_raw_full))
    if n_groups < MIN_GROUPS_FOR_MODEL:
        METHOD_SUMMARY_ROWS.append({"output_name": output_name, "status": "skipped", "reason": f"Not enough groups ({n_groups})",
                                    "output_family": classify_output_family(output_name), "best_y_strategy": "",
                                    "final_method_name": "", "final_model_name": "", "target_transform": "",
                                    "final_chosen_topk": np.nan, "n_cv_runs": 0, "support_share": 0.0,
                                    "cv_r2_mean": np.nan, "cv_r2_std": np.nan, "cv_rmse_mean": np.nan, "cv_mae_mean": np.nan})
        continue

    best_y_strategy = get_best_y_strategy_for_output(output_name, groups=g_raw_full)
    fold_rows = []
    split_iter = repeated_group_splits(g_raw_full, n_splits=min(FINAL_OUTER_SPLITS, n_groups), n_repeats=OUTER_REPEATS,
                                       random_state=RANDOM_STATE, return_split_id=True, test_size=OUTER_TEST_SIZE)

    for cv_run_id, repeat_id, fold_id, tr_idx, te_idx in split_iter:
        try:
            X_tr_raw = X_raw_full.iloc[tr_idx].reset_index(drop=True)
            y_tr_raw = y_raw_full[tr_idx]
            g_tr_raw = g_raw_full[tr_idx]
            xkey_tr = xkey_full[tr_idx]
            X_te_raw = X_raw_full.iloc[te_idx].reset_index(drop=True)
            y_te_raw = y_raw_full[te_idx]
            g_te_raw = g_raw_full[te_idx]
            xkey_te = xkey_full[te_idx]

            rough_cols, _ = prefilter_features_groupwise(X_tr_raw, y_tr_raw, g_tr_raw,
                                                         top_k=get_prefilter_topk_for_output_local(output_name),
                                                         corr_threshold=CORR_PRUNE_THRESHOLD,
                                                         random_state=RANDOM_STATE + cv_run_id)
            if len(rough_cols) < 2:
                continue
            selected_train_df, _ = select_best_y_within_group(X_tr_raw, y_tr_raw, g_tr_raw, xkey_tr,
                                                              rough_cols=rough_cols, strategy=best_y_strategy,
                                                              random_state=RANDOM_STATE + cv_run_id)
            test_group_df = aggregate_test_groups_by_strategy(X_te_raw, y_te_raw, g_te_raw, xkey_te,
                                                              strategy=best_y_strategy)
            selected_train_df = _ensure_standard_cols(selected_train_df, group_col=group_col, xkey_col=xkey_col,
                                                      group_values=g_tr_raw, xkey_values=xkey_tr)
            test_group_df = _ensure_standard_cols(test_group_df, group_col=group_col, xkey_col=xkey_col,
                                                  group_values=g_te_raw, xkey_values=xkey_te)
            selected_train_df = selected_train_df.dropna(subset=["target", "GROUP_ID"]).reset_index(drop=True)
            test_group_df = test_group_df.dropna(subset=["target", "GROUP_ID"]).reset_index(drop=True)
            if selected_train_df.empty or test_group_df.empty:
                continue
            if "XKEY" not in selected_train_df.columns:
                selected_train_df["XKEY"] = [f"TR_{cv_run_id}_{i}" for i in range(len(selected_train_df))]
            X_train_sel = selected_train_df[feature_cols].copy()
            y_train_sel = selected_train_df["target"].to_numpy(dtype=float)
            g_train_sel = selected_train_df["GROUP_ID"].astype(str).to_numpy()
            X_test_group = test_group_df[feature_cols].copy()
            y_test_group = test_group_df["target"].to_numpy(dtype=float)
            g_test_group = test_group_df["GROUP_ID"].astype(str).to_numpy()

            ranked_features, rank_df = build_ranked_features_train(X_train_sel, y_train_sel, g_train_sel, output_name, cv_run_id)
            if len(ranked_features) < 2:
                continue

            for method_name in get_method_candidates_for_output(output_name):
                res = evaluate_method_train_test(method_name, output_name, X_train_sel, y_train_sel, g_train_sel,
                                                 selected_train_df, X_test_group, y_test_group, ranked_features,
                                                 random_state=RANDOM_STATE + cv_run_id)
                if res is None:
                    continue
                fold_rows.append({
                    "output_name": output_name, "cv_run_id": int(cv_run_id), "repeat_id": int(repeat_id), "fold_id": int(fold_id),
                    "output_family": classify_output_family(output_name), "best_y_strategy": best_y_strategy,
                    "method_name": res["method_name"], "model_name": res["model_name"], "target_transform": res["target_transform"],
                    "topk": int(res["topk"]), "n_train_groups": int(len(pd.unique(g_train_sel))), "n_test_groups": int(len(pd.unique(g_test_group))),
                    "n_features": int(res["n_features"]), "selected_features": ", ".join(res["selected_features"]),
                    "outer_r2": float(res["outer_r2"]), "outer_rmse": float(res["outer_rmse"]), "outer_mae": float(res["outer_mae"]),
                    "search_score": float(res["search_score"]), "status": "ok", "reason": ""
                })
        except Exception as e:
            fold_rows.append({"output_name": output_name, "cv_run_id": int(cv_run_id), "repeat_id": int(repeat_id), "fold_id": int(fold_id),
                              "output_family": classify_output_family(output_name), "best_y_strategy": best_y_strategy,
                              "method_name": "error", "model_name": "", "target_transform": "", "topk": np.nan,
                              "n_train_groups": np.nan, "n_test_groups": np.nan, "n_features": 0, "selected_features": "",
                              "outer_r2": np.nan, "outer_rmse": np.nan, "outer_mae": np.nan, "search_score": np.nan,
                              "status": "error", "reason": f"{type(e).__name__}: {e}"})

    fold_df = pd.DataFrame(fold_rows)
    ok_fold_df = fold_df.loc[fold_df["status"] == "ok"].copy()
    out_dir = os.path.join(PER_OUTPUT_DIR, sanitize_filename(output_name))
    os.makedirs(out_dir, exist_ok=True)
    export_df(fold_df, os.path.join(out_dir, "method_comparison_fold_metrics"))

    if ok_fold_df.empty:
        METHOD_SUMMARY_ROWS.append({"output_name": output_name, "status": "failed", "reason": "No valid method comparison result",
                                    "output_family": classify_output_family(output_name), "best_y_strategy": best_y_strategy,
                                    "final_method_name": "", "final_model_name": "", "target_transform": "",
                                    "final_chosen_topk": np.nan, "n_cv_runs": 0, "support_share": 0.0,
                                    "cv_r2_mean": np.nan, "cv_r2_std": np.nan, "cv_rmse_mean": np.nan, "cv_mae_mean": np.nan})
        continue

    max_cv_runs = max(1, ok_fold_df["cv_run_id"].nunique())
    model_summary_df = ok_fold_df.groupby(["method_name", "model_name", "target_transform", "topk"], as_index=False).agg(
        mean_outer_r2=("outer_r2", "mean"), std_outer_r2=("outer_r2", "std"), mean_outer_rmse=("outer_rmse", "mean"),
        mean_outer_mae=("outer_mae", "mean"), n_cv_runs=("cv_run_id", "count"), mean_n_features=("n_features", "mean")
    ).reset_index(drop=True)
    model_summary_df["support_share"] = model_summary_df["n_cv_runs"] / max_cv_runs
    model_summary_df["score_obj"] = model_summary_df["mean_outer_r2"] - 0.12*model_summary_df["std_outer_r2"].fillna(0.0) + 0.10*model_summary_df["support_share"] - 0.01*model_summary_df["topk"].astype(float)
    model_summary_df = model_summary_df.sort_values(["score_obj", "mean_outer_r2", "std_outer_r2", "topk"], ascending=[False, False, True, True]).reset_index(drop=True)
    best_row = model_summary_df.iloc[0]
    METHOD_SUMMARY_ROWS.append({
        "output_name": output_name, "status": "ok", "reason": "", "output_family": classify_output_family(output_name),
        "best_y_strategy": best_y_strategy, "final_method_name": best_row["method_name"], "final_model_name": best_row["model_name"],
        "target_transform": best_row["target_transform"], "final_chosen_topk": int(best_row["topk"]), "n_cv_runs": int(best_row["n_cv_runs"]),
        "support_share": float(best_row["support_share"]), "cv_r2_mean": float(best_row["mean_outer_r2"]),
        "cv_r2_std": float(0.0 if pd.isna(best_row["std_outer_r2"]) else best_row["std_outer_r2"]),
        "cv_rmse_mean": float(best_row["mean_outer_rmse"]), "cv_mae_mean": float(best_row["mean_outer_mae"])
    })
    export_df(model_summary_df, os.path.join(out_dir, "method_comparison_summary"))
    export_df(ok_fold_df, os.path.join(out_dir, "method_comparison_fold_metrics_ok_only"))
    PER_OUTPUT_RESULTS[output_name] = {"fold_df": fold_df, "summary_df": model_summary_df}

METHOD_SUMMARY_DF = pd.DataFrame(METHOD_SUMMARY_ROWS)
for c in ["output_name","status","reason","output_family","best_y_strategy","final_method_name","final_model_name","target_transform","final_chosen_topk","n_cv_runs","support_share","cv_r2_mean","cv_r2_std","cv_rmse_mean","cv_mae_mean"]:
    if c not in METHOD_SUMMARY_DF.columns:
        METHOD_SUMMARY_DF[c] = np.nan
METHOD_FOLD_DF = pd.concat([PER_OUTPUT_RESULTS[k]["fold_df"] for k in PER_OUTPUT_RESULTS], axis=0).reset_index(drop=True) if PER_OUTPUT_RESULTS else pd.DataFrame()
export_df(METHOD_SUMMARY_DF, os.path.join(MODEL_EXPORT_DIR, "all_output_method_comparison_summary"))
if not METHOD_FOLD_DF.empty:
    export_df(METHOD_FOLD_DF, os.path.join(MODEL_EXPORT_DIR, "all_output_method_comparison_folds"))
display(METHOD_SUMMARY_DF.sort_values("cv_r2_mean", ascending=False) if not METHOD_SUMMARY_DF.empty else METHOD_SUMMARY_DF)


,output_name,status,reason,output_family,best_y_strategy,final_method_name,final_model_name,target_transform,final_chosen_topk,n_cv_runs,support_share,cv_r2_mean,cv_r2_std,cv_rmse_mean,cv_mae_mean
14,Vibrational response | FRF (g/N) | 3000-6500 H...,ok,,vibrational,robust_trimmed_median,minimal_class_average,MinimalClassAverage,mixed,4,1,0.083333,0.922416,0.000000,0.048272,0.028940
13,Vibrational response | FRF (g/N) | 300-3000 Hz...,ok,,vibrational,robust_trimmed_median,baseline_stability,Kernel_Ridge_RBF,yeo_johnson,4,3,0.250000,0.920372,0.035034,0.012874,0.008482
8,Thermal characteristics | h | W/m.K,ok,,thermal,singleton_raw,baseline_stability,PCR_Ridge,yeo_johnson,4,1,0.083333,0.773800,0.000000,0.751409,0.602006
15,Vibrational response | FRF (g/N) | 6500-8000 H...,ok,,vibrational,robust_trimmed_median,minimal_class_average,MinimalClassAverage,mixed,6,4,0.333333,0.612181,0.177028,0.358346,0.221848
4,Yield strength,ok,,mechanical_energy,singleton_raw,residual_ridge_extratrees,Residual_Ridge_ExtraTrees,raw,4,2,0.166667,0.563750,0.040436,29.143942,21.632978
12,Vibrational response | FRF (g/N) | 300-8000 Hz...,ok,,vibrational,robust_trimmed_median,baseline_stability,Kernel_Ridge_RBF,yeo_johnson,6,6,0.500000,0.546643,0.224581,0.084631,0.058370
7,Thermal characteristics | Thermal conductivity...,ok,,thermal,singleton_raw,residual_ridge_extratrees,Residual_Ridge_ExtraTrees,yeo_johnson,3,2,0.166667,0.423122,0.076853,0.559626,0.446909
2,APS,ok,,mechanical_energy,singleton_raw,spca_pls,PLS,raw,8,1,0.083333,0.416870,0.000000,76.192754,54.681713
9,Thermal characteristics | Heating rate | °C/s,ok,,thermal,singleton_raw,residual_pls_extratrees,Residual_PLS_ExtraTrees,yeo_johnson,4,1,0.083333,0.414678,0.000000,0.001553,0.001409
3,AS,ok,,mechanical_energy,singleton_raw,spca_pls,PLS,raw,8,1,0.083333,0.412813,0.000000,73.657815,53.065431


In [9]:

# ============================================================
# Cell E1. Refit best method on full data (SAFE VERSION)
# ============================================================

FINAL_REFIT_SUMMARY_ROWS = []
FINAL_MODEL_BACKUP = {}

def _ensure_standard_cols_refit(df_in, *, group_col=None, xkey_col=None, group_values=None, xkey_values=None):
    df = df_in.copy()
    if "GROUP_ID" not in df.columns:
        if group_col is not None and group_col in df.columns:
            df = df.rename(columns={group_col: "GROUP_ID"})
        elif group_values is not None and len(df) == len(group_values):
            df["GROUP_ID"] = group_values[:len(df)]
    if "XKEY" not in df.columns:
        if xkey_col is not None and xkey_col in df.columns:
            df = df.rename(columns={xkey_col: "XKEY"})
        elif xkey_values is not None and len(df) == len(xkey_values):
            df["XKEY"] = xkey_values[:len(df)]
    if "GROUP_ID" in df.columns:
        df["GROUP_ID"] = df["GROUP_ID"].astype(str)
    if "XKEY" in df.columns:
        df["XKEY"] = df["XKEY"].astype(str)
    if "target" in df.columns:
        df["target"] = pd.to_numeric(df["target"], errors="coerce")
    return df

for output_name, bundle in DATA_BY_OUTPUT.items():
    try:
        hit = METHOD_SUMMARY_DF[(METHOD_SUMMARY_DF["output_name"] == output_name) & (METHOD_SUMMARY_DF["status"] == "ok")]
        if hit.empty:
            FINAL_REFIT_SUMMARY_ROWS.append({"output_name": output_name, "status": "skipped", "reason": "No successful method summary row",
                                             "best_y_strategy": "", "final_method_name": "", "final_model_name": "", "target_transform": "",
                                             "final_chosen_topk": np.nan, "n_groups_full": 0, "n_final_features": 0,
                                             "final_train_r2": np.nan, "final_train_rmse": np.nan, "final_train_mae": np.nan})
            continue
        chosen_method = hit.iloc[0]["final_method_name"]
        chosen_topk = int(hit.iloc[0]["final_chosen_topk"]) if pd.notna(hit.iloc[0]["final_chosen_topk"]) else np.nan
        chosen_strategy = hit.iloc[0]["best_y_strategy"]
        chosen_transform = hit.iloc[0]["target_transform"]
        df = bundle["df"].copy(); feature_cols=bundle["feature_cols"]; target_col=bundle["target_col"]; group_col=bundle["group_col"]; xkey_col=bundle["xkey_col"]
        X_raw_full = df[feature_cols].copy(); y_raw_full = pd.to_numeric(df[target_col], errors="coerce").to_numpy(dtype=float); g_raw_full = df[group_col].astype(str).to_numpy(); xkey_full = df[xkey_col].astype(str).to_numpy()
        base_mask = np.isfinite(y_raw_full) & pd.notna(g_raw_full) & pd.notna(xkey_full)
        X_raw_full = X_raw_full.loc[base_mask].reset_index(drop=True); y_raw_full = y_raw_full[base_mask]; g_raw_full = g_raw_full[base_mask]; xkey_full = xkey_full[base_mask]
        rough_cols, _ = prefilter_features_groupwise(X_raw_full, y_raw_full, g_raw_full, top_k=get_prefilter_topk_for_output_local(output_name), corr_threshold=CORR_PRUNE_THRESHOLD, random_state=RANDOM_STATE+999)
        if len(rough_cols) < 2:
            raise RuntimeError("rough_cols < 2 in refit")
        selected_train_df, candidate_df = select_best_y_within_group(X_raw_full, y_raw_full, g_raw_full, xkey_full, rough_cols=rough_cols, strategy=chosen_strategy, random_state=RANDOM_STATE+999)
        selected_train_df = _ensure_standard_cols_refit(selected_train_df, group_col=group_col, xkey_col=xkey_col, group_values=g_raw_full, xkey_values=xkey_full)
        selected_train_df = selected_train_df.dropna(subset=["target","GROUP_ID"]).reset_index(drop=True)
        if selected_train_df.empty:
            raise RuntimeError("selected_train_df empty after cleanup")
        if "XKEY" not in selected_train_df.columns:
            selected_train_df["XKEY"] = [f"FULL_{i}" for i in range(len(selected_train_df))]
        X_train_sel = selected_train_df[feature_cols].copy(); y_train_sel = selected_train_df["target"].to_numpy(dtype=float); g_train_sel = selected_train_df["GROUP_ID"].astype(str).to_numpy()
        ranked_features, rank_df = build_ranked_features_train(X_train_sel, y_train_sel, g_train_sel, output_name, 999)
        if len(ranked_features) < 2:
            raise RuntimeError("ranked_features < 2 in refit")
        res = evaluate_method_train_test(chosen_method, output_name, X_train_sel, y_train_sel, g_train_sel, selected_train_df, X_train_sel.copy(), y_train_sel.copy(), ranked_features, random_state=RANDOM_STATE+999)
        if res is None:
            raise RuntimeError("evaluate_method_train_test returned None in refit")
        feats = list(res["selected_features"]); fitted_estimator = res["fitted_estimator"]
        if chosen_method in ("spca_ridge","spca_huber","spca_pls"):
            Z_full = transform_spca(fitted_estimator["spca_obj"], X_train_sel[ranked_features]); train_pred = np.asarray(fitted_estimator["reg_model"].predict(Z_full)).reshape(-1)
        elif chosen_method == "block_pca_ridge":
            Z_full = transform_block_pca(fitted_estimator["block_obj"], X_train_sel); train_pred = np.asarray(fitted_estimator["reg_model"].predict(Z_full)).reshape(-1)
        elif chosen_method == "minimal_class_average":
            members = fitted_estimator["members"]; weights = np.asarray(fitted_estimator["weights"], dtype=float); preds=[]
            for mem in members:
                preds.append(np.asarray(mem["est"].predict(X_train_sel[mem["feats"]])).reshape(-1))
            train_pred = np.average(np.vstack(preds), axis=0, weights=weights)
        elif chosen_method == "direct_omp_ridge":
            prep=fitted_estimator["prep"]; ridge=fitted_estimator["ridge"]; idx=fitted_estimator["idx"]; train_pred = np.asarray(ridge.predict(prep.transform(X_train_sel[feats])[:, idx])).reshape(-1)
        elif chosen_method in ("direct_gpr_matern","direct_gpr_rq"):
            prep=fitted_estimator["prep"]; gpr=fitted_estimator["gpr"]; train_pred=np.asarray(gpr.predict(prep.transform(X_train_sel[feats]))).reshape(-1)
        elif chosen_method in ("residual_huber_krr","residual_ard_krr"):
            if chosen_method == "residual_huber_krr":
                base_pred = np.asarray(fitted_estimator["base_estimator"].predict(X_train_sel[feats])).reshape(-1)
            else:
                base_pred = np.asarray(fitted_estimator["base_estimator"].predict(X_train_sel[feats])).reshape(-1)
            train_pred = base_pred + fitted_estimator["krr"].predict(fitted_estimator["prep"].transform(X_train_sel[feats]))
        else:
            train_pred = np.asarray(fitted_estimator.predict(X_train_sel[feats])).reshape(-1)
        tr_r2, tr_rmse, tr_mae = score_prediction(y_train_sel, train_pred)
        selected_export_cols = [c for c in ["GROUP_ID","XKEY"] + feats + ["target"] if c in selected_train_df.columns]
        selected_export_df = selected_train_df[selected_export_cols].copy().rename(columns={"target":"selected_target"})
        out_safe = sanitize_filename(output_name)
        export_df(selected_export_df, os.path.join(FINAL_DATA_DIR, f"{out_safe}_final_selected_input_output"))
        export_df(candidate_df, os.path.join(FINAL_DATA_DIR, f"{out_safe}_candidate_repeated_rows"))
        export_df(rank_df, os.path.join(FINAL_DATA_DIR, f"{out_safe}_feature_rank"))
        backup_payload = {"output_name":output_name, "method_name":chosen_method, "model_name":res["model_name"], "target_transform":res["target_transform"], "selected_features":feats, "ranked_features":ranked_features, "fitted_estimator":fitted_estimator, "selected_export_df":selected_export_df}
        backup_path = os.path.join(FINAL_BACKUP_DIR, f"{out_safe}_model_backup.joblib"); joblib.dump(backup_payload, backup_path); FINAL_MODEL_BACKUP[output_name] = backup_payload
        FINAL_REFIT_SUMMARY_ROWS.append({"output_name": output_name, "status": "ok", "reason": "", "best_y_strategy": chosen_strategy,
                                         "final_method_name": chosen_method, "final_model_name": res["model_name"], "target_transform": res["target_transform"],
                                         "final_chosen_topk": int(chosen_topk) if pd.notna(chosen_topk) else np.nan, "n_groups_full": int(len(pd.unique(g_train_sel))),
                                         "n_final_features": int(len(feats)), "final_train_r2": float(tr_r2) if pd.notna(tr_r2) else np.nan,
                                         "final_train_rmse": float(tr_rmse) if pd.notna(tr_rmse) else np.nan, "final_train_mae": float(tr_mae) if pd.notna(tr_mae) else np.nan})
    except Exception as e:
        FINAL_REFIT_SUMMARY_ROWS.append({"output_name": output_name, "status": "error", "reason": f"{type(e).__name__}: {e}",
                                         "best_y_strategy": "", "final_method_name": "", "final_model_name": "", "target_transform": "",
                                         "final_chosen_topk": np.nan, "n_groups_full": 0, "n_final_features": 0,
                                         "final_train_r2": np.nan, "final_train_rmse": np.nan, "final_train_mae": np.nan})

FINAL_REFIT_SUMMARY_DF = pd.DataFrame(FINAL_REFIT_SUMMARY_ROWS)
for c in ["output_name","status","reason","best_y_strategy","final_method_name","final_model_name","target_transform","final_chosen_topk","n_groups_full","n_final_features","final_train_r2","final_train_rmse","final_train_mae"]:
    if c not in FINAL_REFIT_SUMMARY_DF.columns:
        FINAL_REFIT_SUMMARY_DF[c] = np.nan
export_df(FINAL_REFIT_SUMMARY_DF, os.path.join(MODEL_EXPORT_DIR, "final_refit_summary"))
display(FINAL_REFIT_SUMMARY_DF.sort_values("final_train_r2", ascending=False) if not FINAL_REFIT_SUMMARY_DF.empty else FINAL_REFIT_SUMMARY_DF)


,output_name,status,reason,best_y_strategy,final_method_name,final_model_name,target_transform,final_chosen_topk,n_groups_full,n_final_features,final_train_r2,final_train_rmse,final_train_mae
13,Vibrational response | FRF (g/N) | 300-3000 Hz...,ok,,robust_trimmed_median,baseline_stability,Kernel_Ridge_RBF,yeo_johnson,4.0,128,4,0.986459,0.010487,0.004785
14,Vibrational response | FRF (g/N) | 3000-6500 H...,ok,,robust_trimmed_median,minimal_class_average,MinimalClassAverage,mixed,4.0,128,6,0.984826,0.024082,0.015316
12,Vibrational response | FRF (g/N) | 300-8000 Hz...,ok,,robust_trimmed_median,baseline_stability,Kernel_Ridge_RBF,yeo_johnson,6.0,128,6,0.951801,0.027423,0.018787
8,Thermal characteristics | h | W/m.K,ok,,singleton_raw,baseline_stability,Kernel_Ridge_RBF,yeo_johnson,4.0,25,3,0.786517,0.606806,0.470067
4,Yield strength,ok,,singleton_raw,residual_ridge_extratrees,Residual_Ridge_ExtraTrees,yeo_johnson,4.0,56,2,0.536344,32.848450,25.041692
0,Modulus,ok,,singleton_raw,baseline_stability,Ridge,yeo_johnson,2.0,56,4,0.511028,2460.408295,1887.829743
6,Total energy,ok,,singleton_raw,residual_ridge_extratrees,Residual_Ridge_ExtraTrees,yeo_johnson,2.0,56,2,0.503022,45.280947,35.778224
3,AS,ok,,singleton_raw,spca_pls,PLS,yeo_johnson,8.0,56,8,0.398790,89.733608,68.444676
2,APS,ok,,singleton_raw,spca_pls,PLS,yeo_johnson,8.0,56,8,0.396479,92.668813,70.794080
5,Densif. strength,ok,,singleton_raw,baseline_stability,Ridge,yeo_johnson,2.0,56,3,0.385106,189.997737,142.413967


In [10]:

# ============================================================
# Cell F1. Final merged summary
# ============================================================

FINAL_SUMMARY_DF = METHOD_SUMMARY_DF.merge(
    FINAL_REFIT_SUMMARY_DF,
    on="output_name",
    how="left",
    suffixes=("_cv", "_refit")
)

if "cv_r2_mean" not in FINAL_SUMMARY_DF.columns:
    FINAL_SUMMARY_DF["cv_r2_mean"] = np.nan

export_df(FINAL_SUMMARY_DF, os.path.join(MODEL_EXPORT_DIR, "final_summary_merged"))

if not FINAL_SUMMARY_DF.empty:
    display(FINAL_SUMMARY_DF.sort_values("cv_r2_mean", ascending=False))
else:
    display(FINAL_SUMMARY_DF)


,output_name,status_cv,reason_cv,output_family,best_y_strategy_cv,final_method_name_cv,final_model_name_cv,target_transform_cv,final_chosen_topk_cv,n_cv_runs,...,best_y_strategy_refit,final_method_name_refit,final_model_name_refit,target_transform_refit,final_chosen_topk_refit,n_groups_full,n_final_features,final_train_r2,final_train_rmse,final_train_mae
14,Vibrational response | FRF (g/N) | 3000-6500 H...,ok,,vibrational,robust_trimmed_median,minimal_class_average,MinimalClassAverage,mixed,4,1,...,robust_trimmed_median,minimal_class_average,MinimalClassAverage,mixed,4.0,128,6,0.984826,0.024082,0.015316
13,Vibrational response | FRF (g/N) | 300-3000 Hz...,ok,,vibrational,robust_trimmed_median,baseline_stability,Kernel_Ridge_RBF,yeo_johnson,4,3,...,robust_trimmed_median,baseline_stability,Kernel_Ridge_RBF,yeo_johnson,4.0,128,4,0.986459,0.010487,0.004785
8,Thermal characteristics | h | W/m.K,ok,,thermal,singleton_raw,baseline_stability,PCR_Ridge,yeo_johnson,4,1,...,singleton_raw,baseline_stability,Kernel_Ridge_RBF,yeo_johnson,4.0,25,3,0.786517,0.606806,0.470067
15,Vibrational response | FRF (g/N) | 6500-8000 H...,ok,,vibrational,robust_trimmed_median,minimal_class_average,MinimalClassAverage,mixed,6,4,...,,,,,NaN,0,0,NaN,NaN,NaN
4,Yield strength,ok,,mechanical_energy,singleton_raw,residual_ridge_extratrees,Residual_Ridge_ExtraTrees,raw,4,2,...,singleton_raw,residual_ridge_extratrees,Residual_Ridge_ExtraTrees,yeo_johnson,4.0,56,2,0.536344,32.848450,25.041692
12,Vibrational response | FRF (g/N) | 300-8000 Hz...,ok,,vibrational,robust_trimmed_median,baseline_stability,Kernel_Ridge_RBF,yeo_johnson,6,6,...,robust_trimmed_median,baseline_stability,Kernel_Ridge_RBF,yeo_johnson,6.0,128,6,0.951801,0.027423,0.018787
7,Thermal characteristics | Thermal conductivity...,ok,,thermal,singleton_raw,residual_ridge_extratrees,Residual_Ridge_ExtraTrees,yeo_johnson,3,2,...,singleton_raw,residual_ridge_extratrees,Residual_Ridge_ExtraTrees,raw,3.0,65,2,0.350479,1.023268,0.567536
2,APS,ok,,mechanical_energy,singleton_raw,spca_pls,PLS,raw,8,1,...,singleton_raw,spca_pls,PLS,yeo_johnson,8.0,56,8,0.396479,92.668813,70.794080
9,Thermal characteristics | Heating rate | °C/s,ok,,thermal,singleton_raw,residual_pls_extratrees,Residual_PLS_ExtraTrees,yeo_johnson,4,1,...,singleton_raw,residual_pls_extratrees,Residual_PLS_ExtraTrees,raw,4.0,65,3,0.342529,0.001854,0.001470
3,AS,ok,,mechanical_energy,singleton_raw,spca_pls,PLS,raw,8,1,...,singleton_raw,spca_pls,PLS,yeo_johnson,8.0,56,8,0.398790,89.733608,68.444676


In [11]:

# ============================================================
# Cell G1. Backup reload for later plotting / next cell usage
# ============================================================

BACKUP_FILES = sorted(Path(FINAL_BACKUP_DIR).glob("*_model_backup.joblib"))
AVAILABLE_OUTPUTS = [p.name.replace("_model_backup.joblib", "") for p in BACKUP_FILES]

print("Available backups:")
for name in AVAILABLE_OUTPUTS:
    print(" -", name)

BACKUP_MAP = {}
for p in BACKUP_FILES:
    BACKUP_MAP[p.name.replace("_model_backup.joblib", "")] = joblib.load(p)

print("\nUse BACKUP_MAP['<sanitized_output_name>'] in the next cell.")


Available backups:
 - APS
 - AS
 - Com. Strength
 - Densif. strength
 - Modulus
 - Thermal characteristics _ h _ W_m.K
 - Thermal characteristics _ Heating rate _ °C_s
 - Thermal characteristics _ Thermal conductivity _ W_m.K
 - Total energy
 - Vibrational response _ FRF (g_N) _ 300-3000 Hz _ AVG
 - Vibrational response _ FRF (g_N) _ 300-8000 Hz _ AVG
 - Vibrational response _ FRF (g_N) _ 3000-6500 Hz _ AVG
 - Yield strength

Use BACKUP_MAP['<sanitized_output_name>'] in the next cell.


# Additional untried methods extension

This section adds method families not present in the prior three notebooks and reruns the comparison/refit safely.

Added methods:
- `direct_extra_trees_shallow`
- `direct_random_forest_shallow`
- `direct_hist_gb`
- `direct_svr_rbf`
- `direct_svr_linear`
- `direct_knn_distance`

These are appended as overrides so they can be run after the earlier cells.

In [12]:

# ============================================================
# Extra untried methods: shallow ensembles / SVR / KNN
# ============================================================
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor

def fit_predict_direct_estimator(X_train, y_train, X_test, estimator, y_transform="raw"):
    steps = [("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]
    pipe = Pipeline(steps + [("model", estimator)])
    ytfm = make_y_transformer(y_transform) if "make_y_transformer" in globals() else None
    if ytfm is not None:
        est = TransformedTargetRegressor(regressor=pipe, transformer=ytfm)
    else:
        est = pipe
    est.fit(X_train, y_train)
    pred = np.asarray(est.predict(X_test)).reshape(-1)
    return est, pred

def get_method_candidates_for_output(output_name):
    fam = classify_output_family(output_name)
    methods = [
        "baseline_stability",
        "stability_lasso_ridge",
        "spca_ridge",
        "spca_huber",
        "block_pca_ridge",
        "bagged_subspace_ridge",
        "minimal_class_average",
        "direct_ard",
        "direct_elasticnet_cv",
        "direct_omp_ridge",
        "direct_quantile_median",
        "direct_theilsen",
        "direct_ransac_ridge",
        "direct_gpr_matern",
        "direct_gpr_rq",
        "residual_huber_krr",
        "residual_ard_krr",
        "residual_ridge_extratrees",
        "residual_pls_extratrees",
        # newly added
        "direct_extra_trees_shallow",
        "direct_random_forest_shallow",
        "direct_hist_gb",
        "direct_svr_rbf",
        "direct_svr_linear",
        "direct_knn_distance",
    ]
    if fam in ("thermal", "vibrational"):
        methods += ["multitask_screen_ridge", "multitask_screen_pls"]
    if fam == "mechanical_energy":
        methods += ["spca_pls"]
    return methods

_prev_eval_method_train_test = evaluate_method_train_test

def _pick_best_direct_candidate(cands):
    if not cands:
        return None
    cdf = pd.DataFrame([{
        "score_obj": c["score_obj"],
        "n_features": c["n_features"],
        "complexity_rank": c["complexity_rank"]
    } for c in cands])
    idx = choose_within_tolerance(cdf.reset_index())
    if idx is None:
        return cands[int(cdf["score_obj"].idxmax())]
    # choose_within_tolerance returns row dict containing original index if reset_index used
    chosen_index = int(idx["index"]) if "index" in idx else int(cdf["score_obj"].idxmax())
    return cands[chosen_index]

def evaluate_method_train_test(method_name, output_name, X_train_sel, y_train_sel, g_train_sel, selected_train_df,
                               X_test_group, y_test_group, ranked_features, random_state=42):
    fam = classify_output_family(output_name)

    # ---- NEW METHODS FIRST ----
    if method_name == "direct_extra_trees_shallow":
        base_feats = ranked_features[:min(len(ranked_features), 10 if fam!="mechanical_energy" else 8)]
        if len(base_feats) < 2:
            return None
        cands = []
        for topk in [2, 3, 4, 6]:
            feats = base_feats[:min(topk, len(base_feats))]
            for tt in TARGET_TRANSFORM_CANDIDATES:
                est_model = ExtraTreesRegressor(
                    n_estimators=180,
                    max_depth=3,
                    min_samples_leaf=3,
                    min_samples_split=4,
                    random_state=random_state,
                    n_jobs=-1,
                )
                try:
                    inner = inner_group_cv_score_direct(
                        X_train_sel[feats], y_train_sel, g_train_sel, est_model, y_transform=tt,
                        n_splits=5, n_repeats=1, random_state=random_state
                    )
                except NameError:
                    # local fallback if helper absent
                    inner = {"mean_r2": np.nan, "std_r2": np.nan}
                score_obj = (inner.get("mean_r2", np.nan) if pd.notna(inner.get("mean_r2", np.nan)) else -1e9) - 0.15*(0 if pd.isna(inner.get("std_r2", np.nan)) else inner["std_r2"]) - 0.01*len(feats)
                cands.append({"feats":feats,"tt":tt,"est_model":est_model,"inner":inner,"score_obj":score_obj,"n_features":len(feats),"complexity_rank":6})
        best = _pick_best_direct_candidate(cands)
        if best is None or not np.isfinite(best["score_obj"]):
            return None
        est, pred = fit_predict_direct_estimator(X_train_sel[best["feats"]], y_train_sel, X_test_group[best["feats"]], best["est_model"], y_transform=best["tt"])
        r2, rmse, mae = score_prediction(y_test_group, pred)
        if not np.isfinite(r2): return None
        return {"method_name":method_name,"model_name":"ExtraTrees_Shallow","target_transform":best["tt"],"topk":len(best["feats"]),
                "selected_features":best["feats"],"n_features":len(best["feats"]),"complexity_rank":6,
                "search_score":float(best["inner"].get("mean_r2", np.nan)),"outer_r2":r2,"outer_rmse":rmse,"outer_mae":mae,"fitted_estimator":est}

    if method_name == "direct_random_forest_shallow":
        base_feats = ranked_features[:min(len(ranked_features), 10 if fam!="mechanical_energy" else 8)]
        if len(base_feats) < 2:
            return None
        cands = []
        for topk in [2, 3, 4, 6]:
            feats = base_feats[:min(topk, len(base_feats))]
            for tt in TARGET_TRANSFORM_CANDIDATES:
                est_model = RandomForestRegressor(
                    n_estimators=220,
                    max_depth=4,
                    min_samples_leaf=3,
                    min_samples_split=4,
                    random_state=random_state,
                    n_jobs=-1,
                )
                inner = inner_group_cv_score_direct(X_train_sel[feats], y_train_sel, g_train_sel, est_model, y_transform=tt,
                                                    n_splits=5, n_repeats=1, random_state=random_state)
                score_obj = (inner.get("mean_r2", np.nan) if pd.notna(inner.get("mean_r2", np.nan)) else -1e9) - 0.15*(0 if pd.isna(inner.get("std_r2", np.nan)) else inner["std_r2"]) - 0.01*len(feats)
                cands.append({"feats":feats,"tt":tt,"est_model":est_model,"inner":inner,"score_obj":score_obj,"n_features":len(feats),"complexity_rank":6})
        best = _pick_best_direct_candidate(cands)
        if best is None or not np.isfinite(best["score_obj"]):
            return None
        est, pred = fit_predict_direct_estimator(X_train_sel[best["feats"]], y_train_sel, X_test_group[best["feats"]], best["est_model"], y_transform=best["tt"])
        r2, rmse, mae = score_prediction(y_test_group, pred)
        if not np.isfinite(r2): return None
        return {"method_name":method_name,"model_name":"RandomForest_Shallow","target_transform":best["tt"],"topk":len(best["feats"]),
                "selected_features":best["feats"],"n_features":len(best["feats"]),"complexity_rank":6,
                "search_score":float(best["inner"].get("mean_r2", np.nan)),"outer_r2":r2,"outer_rmse":rmse,"outer_mae":mae,"fitted_estimator":est}

    if method_name == "direct_hist_gb":
        base_feats = ranked_features[:min(len(ranked_features), 10)]
        if len(base_feats) < 2: return None
        cands = []
        for topk in [2,3,4,6]:
            feats = base_feats[:min(topk, len(base_feats))]
            for tt in TARGET_TRANSFORM_CANDIDATES:
                est_model = HistGradientBoostingRegressor(
                    max_depth=3,
                    learning_rate=0.04,
                    max_iter=180,
                    min_samples_leaf=4,
                    l2_regularization=0.2,
                    random_state=random_state,
                )
                inner = inner_group_cv_score_direct(X_train_sel[feats], y_train_sel, g_train_sel, est_model, y_transform=tt,
                                                    n_splits=5, n_repeats=1, random_state=random_state)
                score_obj = (inner.get("mean_r2", np.nan) if pd.notna(inner.get("mean_r2", np.nan)) else -1e9) - 0.16*(0 if pd.isna(inner.get("std_r2", np.nan)) else inner["std_r2"]) - 0.012*len(feats)
                cands.append({"feats":feats,"tt":tt,"est_model":est_model,"inner":inner,"score_obj":score_obj,"n_features":len(feats),"complexity_rank":7})
        best = _pick_best_direct_candidate(cands)
        if best is None or not np.isfinite(best["score_obj"]): return None
        est, pred = fit_predict_direct_estimator(X_train_sel[best["feats"]], y_train_sel, X_test_group[best["feats"]], best["est_model"], y_transform=best["tt"])
        r2, rmse, mae = score_prediction(y_test_group, pred)
        if not np.isfinite(r2): return None
        return {"method_name":method_name,"model_name":"HistGradientBoosting","target_transform":best["tt"],"topk":len(best["feats"]),
                "selected_features":best["feats"],"n_features":len(best["feats"]),"complexity_rank":7,
                "search_score":float(best["inner"].get("mean_r2", np.nan)),"outer_r2":r2,"outer_rmse":rmse,"outer_mae":mae,"fitted_estimator":est}

    if method_name == "direct_svr_rbf":
        base_feats = ranked_features[:min(len(ranked_features), 8)]
        if len(base_feats) < 2: return None
        cands = []
        for topk in [2,3,4,6]:
            feats = base_feats[:min(topk, len(base_feats))]
            for tt in TARGET_TRANSFORM_CANDIDATES:
                for C in [0.5, 1.0, 3.0]:
                    for eps in [0.02, 0.05, 0.1]:
                        est_model = SVR(kernel="rbf", C=C, epsilon=eps, gamma="scale")
                        inner = inner_group_cv_score_direct(X_train_sel[feats], y_train_sel, g_train_sel, est_model, y_transform=tt,
                                                            n_splits=5, n_repeats=1, random_state=random_state)
                        score_obj = (inner.get("mean_r2", np.nan) if pd.notna(inner.get("mean_r2", np.nan)) else -1e9) - 0.14*(0 if pd.isna(inner.get("std_r2", np.nan)) else inner["std_r2"]) - 0.01*len(feats)
                        cands.append({"feats":feats,"tt":tt,"est_model":est_model,"inner":inner,"score_obj":score_obj,"n_features":len(feats),"complexity_rank":6})
        best = _pick_best_direct_candidate(cands)
        if best is None or not np.isfinite(best["score_obj"]): return None
        est, pred = fit_predict_direct_estimator(X_train_sel[best["feats"]], y_train_sel, X_test_group[best["feats"]], best["est_model"], y_transform=best["tt"])
        r2, rmse, mae = score_prediction(y_test_group, pred)
        if not np.isfinite(r2): return None
        return {"method_name":method_name,"model_name":"SVR_RBF_Direct","target_transform":best["tt"],"topk":len(best["feats"]),
                "selected_features":best["feats"],"n_features":len(best["feats"]),"complexity_rank":6,
                "search_score":float(best["inner"].get("mean_r2", np.nan)),"outer_r2":r2,"outer_rmse":rmse,"outer_mae":mae,"fitted_estimator":est}

    if method_name == "direct_svr_linear":
        base_feats = ranked_features[:min(len(ranked_features), 8)]
        if len(base_feats) < 2: return None
        cands = []
        for topk in [2,3,4,6]:
            feats = base_feats[:min(topk, len(base_feats))]
            for tt in TARGET_TRANSFORM_CANDIDATES:
                for C in [0.5, 1.0, 3.0]:
                    for eps in [0.02, 0.05, 0.1]:
                        est_model = SVR(kernel="linear", C=C, epsilon=eps)
                        inner = inner_group_cv_score_direct(X_train_sel[feats], y_train_sel, g_train_sel, est_model, y_transform=tt,
                                                            n_splits=5, n_repeats=1, random_state=random_state)
                        score_obj = (inner.get("mean_r2", np.nan) if pd.notna(inner.get("mean_r2", np.nan)) else -1e9) - 0.12*(0 if pd.isna(inner.get("std_r2", np.nan)) else inner["std_r2"]) - 0.01*len(feats)
                        cands.append({"feats":feats,"tt":tt,"est_model":est_model,"inner":inner,"score_obj":score_obj,"n_features":len(feats),"complexity_rank":5})
        best = _pick_best_direct_candidate(cands)
        if best is None or not np.isfinite(best["score_obj"]): return None
        est, pred = fit_predict_direct_estimator(X_train_sel[best["feats"]], y_train_sel, X_test_group[best["feats"]], best["est_model"], y_transform=best["tt"])
        r2, rmse, mae = score_prediction(y_test_group, pred)
        if not np.isfinite(r2): return None
        return {"method_name":method_name,"model_name":"SVR_Linear_Direct","target_transform":best["tt"],"topk":len(best["feats"]),
                "selected_features":best["feats"],"n_features":len(best["feats"]),"complexity_rank":5,
                "search_score":float(best["inner"].get("mean_r2", np.nan)),"outer_r2":r2,"outer_rmse":rmse,"outer_mae":mae,"fitted_estimator":est}

    if method_name == "direct_knn_distance":
        base_feats = ranked_features[:min(len(ranked_features), 8)]
        if len(base_feats) < 2: return None
        cands = []
        for topk in [2,3,4,6]:
            feats = base_feats[:min(topk, len(base_feats))]
            for tt in TARGET_TRANSFORM_CANDIDATES:
                for k in [3,5,7]:
                    est_model = KNeighborsRegressor(n_neighbors=k, weights="distance", p=2)
                    inner = inner_group_cv_score_direct(X_train_sel[feats], y_train_sel, g_train_sel, est_model, y_transform=tt,
                                                        n_splits=5, n_repeats=1, random_state=random_state)
                    score_obj = (inner.get("mean_r2", np.nan) if pd.notna(inner.get("mean_r2", np.nan)) else -1e9) - 0.16*(0 if pd.isna(inner.get("std_r2", np.nan)) else inner["std_r2"]) - 0.01*len(feats)
                    cands.append({"feats":feats,"tt":tt,"est_model":est_model,"inner":inner,"score_obj":score_obj,"n_features":len(feats),"complexity_rank":6})
        best = _pick_best_direct_candidate(cands)
        if best is None or not np.isfinite(best["score_obj"]): return None
        est, pred = fit_predict_direct_estimator(X_train_sel[best["feats"]], y_train_sel, X_test_group[best["feats"]], best["est_model"], y_transform=best["tt"])
        r2, rmse, mae = score_prediction(y_test_group, pred)
        if not np.isfinite(r2): return None
        return {"method_name":method_name,"model_name":"KNN_Distance_Direct","target_transform":best["tt"],"topk":len(best["feats"]),
                "selected_features":best["feats"],"n_features":len(best["feats"]),"complexity_rank":6,
                "search_score":float(best["inner"].get("mean_r2", np.nan)),"outer_r2":r2,"outer_rmse":rmse,"outer_mae":mae,"fitted_estimator":est}

    # fallback to previous implementation
    return _prev_eval_method_train_test(
        method_name=method_name,
        output_name=output_name,
        X_train_sel=X_train_sel,
        y_train_sel=y_train_sel,
        g_train_sel=g_train_sel,
        selected_train_df=selected_train_df,
        X_test_group=X_test_group,
        y_test_group=y_test_group,
        ranked_features=ranked_features,
        random_state=random_state
    )

def inner_group_cv_score_direct(X, y, groups, estimator, y_transform="raw",
                                n_splits=5, n_repeats=1, random_state=42):
    rows = []
    split_iter = repeated_group_splits(
        groups,
        n_splits=min(n_splits, len(pd.unique(groups))),
        n_repeats=n_repeats,
        random_state=random_state,
        return_split_id=True,
        test_size=INNER_GSS_TEST_SIZE
    )
    for cv_run_id, repeat_id, fold_id, tr_idx, te_idx in split_iter:
        Xtr = X.iloc[tr_idx].copy()
        Xte = X.iloc[te_idx].copy()
        ytr = np.asarray(y)[tr_idx]
        yte = np.asarray(y)[te_idx]
        train_mask = np.isfinite(ytr)
        test_mask = np.isfinite(yte)
        if train_mask.sum() < MIN_VALID_EVAL_SAMPLES or test_mask.sum() < MIN_VALID_EVAL_SAMPLES:
            continue
        Xtr = Xtr.loc[train_mask].reset_index(drop=True)
        Xte = Xte.loc[test_mask].reset_index(drop=True)
        ytr = ytr[train_mask]
        yte = yte[test_mask]
        try:
            est, pred = fit_predict_direct_estimator(Xtr, ytr, Xte, estimator, y_transform=y_transform)
            r2, rmse, mae = score_prediction(yte, pred)
            if np.isfinite(r2):
                rows.append({"r2": r2, "rmse": rmse, "mae": mae})
        except Exception:
            continue
    if not rows:
        return {"mean_r2": np.nan, "std_r2": np.nan, "mean_rmse": np.nan, "mean_mae": np.nan}
    df = pd.DataFrame(rows)
    return {
        "mean_r2": float(df["r2"].mean()),
        "std_r2": float(0.0 if len(df)==1 else df["r2"].std(ddof=1)),
        "mean_rmse": float(df["rmse"].mean()),
        "mean_mae": float(df["mae"].mean()),
    }


In [13]:

# ============================================================
# Rerun D1 and E1 after extra untried-method overrides
# ============================================================
print("Extra untried methods registered. Re-run the next D1/E1 cells below in this notebook to compare them.")


Extra untried methods registered. Re-run the next D1/E1 cells below in this notebook to compare them.


In [14]:

# ============================================================
# Cell D1. Method-comparison outer CV (SAFE VERSION)
# ============================================================

METHOD_SUMMARY_ROWS = []
PER_OUTPUT_RESULTS = {}

def _ensure_standard_cols(df_in, group_col=None, xkey_col=None, group_values=None, xkey_values=None):
    df = df_in.copy()
    if "GROUP_ID" not in df.columns:
        if group_col is not None and group_col in df.columns:
            df = df.rename(columns={group_col: "GROUP_ID"})
        elif group_values is not None and len(df) == len(group_values):
            df["GROUP_ID"] = group_values[:len(df)]
    if "XKEY" not in df.columns:
        if xkey_col is not None and xkey_col in df.columns:
            df = df.rename(columns={xkey_col: "XKEY"})
        elif xkey_values is not None and len(df) == len(xkey_values):
            df["XKEY"] = xkey_values[:len(df)]
    if "GROUP_ID" in df.columns:
        df["GROUP_ID"] = df["GROUP_ID"].astype(str)
    if "XKEY" in df.columns:
        df["XKEY"] = df["XKEY"].astype(str)
    if "target" in df.columns:
        df["target"] = pd.to_numeric(df["target"], errors="coerce")
    return df

for output_name, bundle in DATA_BY_OUTPUT.items():
    df = bundle["df"].copy()
    feature_cols = bundle["feature_cols"]
    target_col = bundle["target_col"]
    group_col = bundle["group_col"]
    xkey_col = bundle["xkey_col"]

    X_raw_full = df[feature_cols].copy()
    y_raw_full = pd.to_numeric(df[target_col], errors="coerce").to_numpy(dtype=float)
    g_raw_full = df[group_col].astype(str).to_numpy()
    xkey_full = df[xkey_col].astype(str).to_numpy()

    base_mask = np.isfinite(y_raw_full) & pd.notna(g_raw_full) & pd.notna(xkey_full)
    X_raw_full = X_raw_full.loc[base_mask].reset_index(drop=True)
    y_raw_full = y_raw_full[base_mask]
    g_raw_full = g_raw_full[base_mask]
    xkey_full = xkey_full[base_mask]

    n_groups = len(pd.unique(g_raw_full))
    if n_groups < MIN_GROUPS_FOR_MODEL:
        METHOD_SUMMARY_ROWS.append({"output_name": output_name, "status": "skipped", "reason": f"Not enough groups ({n_groups})",
                                    "output_family": classify_output_family(output_name), "best_y_strategy": "",
                                    "final_method_name": "", "final_model_name": "", "target_transform": "",
                                    "final_chosen_topk": np.nan, "n_cv_runs": 0, "support_share": 0.0,
                                    "cv_r2_mean": np.nan, "cv_r2_std": np.nan, "cv_rmse_mean": np.nan, "cv_mae_mean": np.nan})
        continue

    best_y_strategy = get_best_y_strategy_for_output(output_name, groups=g_raw_full)
    fold_rows = []
    split_iter = repeated_group_splits(g_raw_full, n_splits=min(FINAL_OUTER_SPLITS, n_groups), n_repeats=OUTER_REPEATS,
                                       random_state=RANDOM_STATE, return_split_id=True, test_size=OUTER_TEST_SIZE)

    for cv_run_id, repeat_id, fold_id, tr_idx, te_idx in split_iter:
        try:
            X_tr_raw = X_raw_full.iloc[tr_idx].reset_index(drop=True)
            y_tr_raw = y_raw_full[tr_idx]
            g_tr_raw = g_raw_full[tr_idx]
            xkey_tr = xkey_full[tr_idx]
            X_te_raw = X_raw_full.iloc[te_idx].reset_index(drop=True)
            y_te_raw = y_raw_full[te_idx]
            g_te_raw = g_raw_full[te_idx]
            xkey_te = xkey_full[te_idx]

            rough_cols, _ = prefilter_features_groupwise(X_tr_raw, y_tr_raw, g_tr_raw,
                                                         top_k=get_prefilter_topk_for_output_local(output_name),
                                                         corr_threshold=CORR_PRUNE_THRESHOLD,
                                                         random_state=RANDOM_STATE + cv_run_id)
            if len(rough_cols) < 2:
                continue
            selected_train_df, _ = select_best_y_within_group(X_tr_raw, y_tr_raw, g_tr_raw, xkey_tr,
                                                              rough_cols=rough_cols, strategy=best_y_strategy,
                                                              random_state=RANDOM_STATE + cv_run_id)
            test_group_df = aggregate_test_groups_by_strategy(X_te_raw, y_te_raw, g_te_raw, xkey_te,
                                                              strategy=best_y_strategy)
            selected_train_df = _ensure_standard_cols(selected_train_df, group_col=group_col, xkey_col=xkey_col,
                                                      group_values=g_tr_raw, xkey_values=xkey_tr)
            test_group_df = _ensure_standard_cols(test_group_df, group_col=group_col, xkey_col=xkey_col,
                                                  group_values=g_te_raw, xkey_values=xkey_te)
            selected_train_df = selected_train_df.dropna(subset=["target", "GROUP_ID"]).reset_index(drop=True)
            test_group_df = test_group_df.dropna(subset=["target", "GROUP_ID"]).reset_index(drop=True)
            if selected_train_df.empty or test_group_df.empty:
                continue
            if "XKEY" not in selected_train_df.columns:
                selected_train_df["XKEY"] = [f"TR_{cv_run_id}_{i}" for i in range(len(selected_train_df))]
            X_train_sel = selected_train_df[feature_cols].copy()
            y_train_sel = selected_train_df["target"].to_numpy(dtype=float)
            g_train_sel = selected_train_df["GROUP_ID"].astype(str).to_numpy()
            X_test_group = test_group_df[feature_cols].copy()
            y_test_group = test_group_df["target"].to_numpy(dtype=float)
            g_test_group = test_group_df["GROUP_ID"].astype(str).to_numpy()

            ranked_features, rank_df = build_ranked_features_train(X_train_sel, y_train_sel, g_train_sel, output_name, cv_run_id)
            if len(ranked_features) < 2:
                continue

            for method_name in get_method_candidates_for_output(output_name):
                res = evaluate_method_train_test(method_name, output_name, X_train_sel, y_train_sel, g_train_sel,
                                                 selected_train_df, X_test_group, y_test_group, ranked_features,
                                                 random_state=RANDOM_STATE + cv_run_id)
                if res is None:
                    continue
                fold_rows.append({
                    "output_name": output_name, "cv_run_id": int(cv_run_id), "repeat_id": int(repeat_id), "fold_id": int(fold_id),
                    "output_family": classify_output_family(output_name), "best_y_strategy": best_y_strategy,
                    "method_name": res["method_name"], "model_name": res["model_name"], "target_transform": res["target_transform"],
                    "topk": int(res["topk"]), "n_train_groups": int(len(pd.unique(g_train_sel))), "n_test_groups": int(len(pd.unique(g_test_group))),
                    "n_features": int(res["n_features"]), "selected_features": ", ".join(res["selected_features"]),
                    "outer_r2": float(res["outer_r2"]), "outer_rmse": float(res["outer_rmse"]), "outer_mae": float(res["outer_mae"]),
                    "search_score": float(res["search_score"]), "status": "ok", "reason": ""
                })
        except Exception as e:
            fold_rows.append({"output_name": output_name, "cv_run_id": int(cv_run_id), "repeat_id": int(repeat_id), "fold_id": int(fold_id),
                              "output_family": classify_output_family(output_name), "best_y_strategy": best_y_strategy,
                              "method_name": "error", "model_name": "", "target_transform": "", "topk": np.nan,
                              "n_train_groups": np.nan, "n_test_groups": np.nan, "n_features": 0, "selected_features": "",
                              "outer_r2": np.nan, "outer_rmse": np.nan, "outer_mae": np.nan, "search_score": np.nan,
                              "status": "error", "reason": f"{type(e).__name__}: {e}"})

    fold_df = pd.DataFrame(fold_rows)
    ok_fold_df = fold_df.loc[fold_df["status"] == "ok"].copy()
    out_dir = os.path.join(PER_OUTPUT_DIR, sanitize_filename(output_name))
    os.makedirs(out_dir, exist_ok=True)
    export_df(fold_df, os.path.join(out_dir, "method_comparison_fold_metrics"))

    if ok_fold_df.empty:
        METHOD_SUMMARY_ROWS.append({"output_name": output_name, "status": "failed", "reason": "No valid method comparison result",
                                    "output_family": classify_output_family(output_name), "best_y_strategy": best_y_strategy,
                                    "final_method_name": "", "final_model_name": "", "target_transform": "",
                                    "final_chosen_topk": np.nan, "n_cv_runs": 0, "support_share": 0.0,
                                    "cv_r2_mean": np.nan, "cv_r2_std": np.nan, "cv_rmse_mean": np.nan, "cv_mae_mean": np.nan})
        continue

    max_cv_runs = max(1, ok_fold_df["cv_run_id"].nunique())
    model_summary_df = ok_fold_df.groupby(["method_name", "model_name", "target_transform", "topk"], as_index=False).agg(
        mean_outer_r2=("outer_r2", "mean"), std_outer_r2=("outer_r2", "std"), mean_outer_rmse=("outer_rmse", "mean"),
        mean_outer_mae=("outer_mae", "mean"), n_cv_runs=("cv_run_id", "count"), mean_n_features=("n_features", "mean")
    ).reset_index(drop=True)
    model_summary_df["support_share"] = model_summary_df["n_cv_runs"] / max_cv_runs
    model_summary_df["score_obj"] = model_summary_df["mean_outer_r2"] - 0.12*model_summary_df["std_outer_r2"].fillna(0.0) + 0.10*model_summary_df["support_share"] - 0.01*model_summary_df["topk"].astype(float)
    model_summary_df = model_summary_df.sort_values(["score_obj", "mean_outer_r2", "std_outer_r2", "topk"], ascending=[False, False, True, True]).reset_index(drop=True)
    best_row = model_summary_df.iloc[0]
    METHOD_SUMMARY_ROWS.append({
        "output_name": output_name, "status": "ok", "reason": "", "output_family": classify_output_family(output_name),
        "best_y_strategy": best_y_strategy, "final_method_name": best_row["method_name"], "final_model_name": best_row["model_name"],
        "target_transform": best_row["target_transform"], "final_chosen_topk": int(best_row["topk"]), "n_cv_runs": int(best_row["n_cv_runs"]),
        "support_share": float(best_row["support_share"]), "cv_r2_mean": float(best_row["mean_outer_r2"]),
        "cv_r2_std": float(0.0 if pd.isna(best_row["std_outer_r2"]) else best_row["std_outer_r2"]),
        "cv_rmse_mean": float(best_row["mean_outer_rmse"]), "cv_mae_mean": float(best_row["mean_outer_mae"])
    })
    export_df(model_summary_df, os.path.join(out_dir, "method_comparison_summary"))
    export_df(ok_fold_df, os.path.join(out_dir, "method_comparison_fold_metrics_ok_only"))
    PER_OUTPUT_RESULTS[output_name] = {"fold_df": fold_df, "summary_df": model_summary_df}

METHOD_SUMMARY_DF = pd.DataFrame(METHOD_SUMMARY_ROWS)
for c in ["output_name","status","reason","output_family","best_y_strategy","final_method_name","final_model_name","target_transform","final_chosen_topk","n_cv_runs","support_share","cv_r2_mean","cv_r2_std","cv_rmse_mean","cv_mae_mean"]:
    if c not in METHOD_SUMMARY_DF.columns:
        METHOD_SUMMARY_DF[c] = np.nan
METHOD_FOLD_DF = pd.concat([PER_OUTPUT_RESULTS[k]["fold_df"] for k in PER_OUTPUT_RESULTS], axis=0).reset_index(drop=True) if PER_OUTPUT_RESULTS else pd.DataFrame()
export_df(METHOD_SUMMARY_DF, os.path.join(MODEL_EXPORT_DIR, "all_output_method_comparison_summary"))
if not METHOD_FOLD_DF.empty:
    export_df(METHOD_FOLD_DF, os.path.join(MODEL_EXPORT_DIR, "all_output_method_comparison_folds"))
display(METHOD_SUMMARY_DF.sort_values("cv_r2_mean", ascending=False) if not METHOD_SUMMARY_DF.empty else METHOD_SUMMARY_DF)


,output_name,status,reason,output_family,best_y_strategy,final_method_name,final_model_name,target_transform,final_chosen_topk,n_cv_runs,support_share,cv_r2_mean,cv_r2_std,cv_rmse_mean,cv_mae_mean
14,Vibrational response | FRF (g/N) | 3000-6500 H...,ok,,vibrational,robust_trimmed_median,minimal_class_average,MinimalClassAverage,mixed,4,1,0.083333,0.922416,0.000000,0.048272,0.028940
13,Vibrational response | FRF (g/N) | 300-3000 Hz...,ok,,vibrational,robust_trimmed_median,baseline_stability,Kernel_Ridge_RBF,yeo_johnson,4,3,0.250000,0.920372,0.035034,0.012874,0.008482
8,Thermal characteristics | h | W/m.K,ok,,thermal,singleton_raw,baseline_stability,PCR_Ridge,yeo_johnson,4,1,0.083333,0.773800,0.000000,0.751409,0.602006
15,Vibrational response | FRF (g/N) | 6500-8000 H...,ok,,vibrational,robust_trimmed_median,minimal_class_average,MinimalClassAverage,mixed,6,4,0.333333,0.612181,0.177028,0.358346,0.221848
4,Yield strength,ok,,mechanical_energy,singleton_raw,direct_ard,ARD_Regression,raw,4,2,0.166667,0.566224,0.067712,29.149976,22.583347
12,Vibrational response | FRF (g/N) | 300-8000 Hz...,ok,,vibrational,robust_trimmed_median,baseline_stability,Kernel_Ridge_RBF,yeo_johnson,6,6,0.500000,0.546643,0.224581,0.084631,0.058370
7,Thermal characteristics | Thermal conductivity...,ok,,thermal,singleton_raw,direct_omp_ridge,OMP_then_Ridge,yeo_johnson,6,1,0.083333,0.434623,0.000000,0.637762,0.450892
3,AS,ok,,mechanical_energy,singleton_raw,block_pca_ridge,Ridge,yeo_johnson,11,1,0.083333,0.405157,0.000000,116.249792,96.406378
6,Total energy,ok,,mechanical_energy,singleton_raw,direct_theilsen,TheilSen,raw,2,1,0.083333,0.390032,0.000000,44.105648,37.787527
10,Thermal characteristics | Cooling rate | °C/s,ok,,thermal,singleton_raw,direct_omp_ridge,OMP_then_Ridge,raw,8,1,0.083333,0.384319,0.000000,0.161535,0.135342


In [15]:

# ============================================================
# Cell E1. Refit best method on full data (SAFE VERSION)
# ============================================================

FINAL_REFIT_SUMMARY_ROWS = []
FINAL_MODEL_BACKUP = {}

def _ensure_standard_cols_refit(df_in, *, group_col=None, xkey_col=None, group_values=None, xkey_values=None):
    df = df_in.copy()
    if "GROUP_ID" not in df.columns:
        if group_col is not None and group_col in df.columns:
            df = df.rename(columns={group_col: "GROUP_ID"})
        elif group_values is not None and len(df) == len(group_values):
            df["GROUP_ID"] = group_values[:len(df)]
    if "XKEY" not in df.columns:
        if xkey_col is not None and xkey_col in df.columns:
            df = df.rename(columns={xkey_col: "XKEY"})
        elif xkey_values is not None and len(df) == len(xkey_values):
            df["XKEY"] = xkey_values[:len(df)]
    if "GROUP_ID" in df.columns:
        df["GROUP_ID"] = df["GROUP_ID"].astype(str)
    if "XKEY" in df.columns:
        df["XKEY"] = df["XKEY"].astype(str)
    if "target" in df.columns:
        df["target"] = pd.to_numeric(df["target"], errors="coerce")
    return df

for output_name, bundle in DATA_BY_OUTPUT.items():
    try:
        hit = METHOD_SUMMARY_DF[(METHOD_SUMMARY_DF["output_name"] == output_name) & (METHOD_SUMMARY_DF["status"] == "ok")]
        if hit.empty:
            FINAL_REFIT_SUMMARY_ROWS.append({"output_name": output_name, "status": "skipped", "reason": "No successful method summary row",
                                             "best_y_strategy": "", "final_method_name": "", "final_model_name": "", "target_transform": "",
                                             "final_chosen_topk": np.nan, "n_groups_full": 0, "n_final_features": 0,
                                             "final_train_r2": np.nan, "final_train_rmse": np.nan, "final_train_mae": np.nan})
            continue
        chosen_method = hit.iloc[0]["final_method_name"]
        chosen_topk = int(hit.iloc[0]["final_chosen_topk"]) if pd.notna(hit.iloc[0]["final_chosen_topk"]) else np.nan
        chosen_strategy = hit.iloc[0]["best_y_strategy"]
        chosen_transform = hit.iloc[0]["target_transform"]
        df = bundle["df"].copy(); feature_cols=bundle["feature_cols"]; target_col=bundle["target_col"]; group_col=bundle["group_col"]; xkey_col=bundle["xkey_col"]
        X_raw_full = df[feature_cols].copy(); y_raw_full = pd.to_numeric(df[target_col], errors="coerce").to_numpy(dtype=float); g_raw_full = df[group_col].astype(str).to_numpy(); xkey_full = df[xkey_col].astype(str).to_numpy()
        base_mask = np.isfinite(y_raw_full) & pd.notna(g_raw_full) & pd.notna(xkey_full)
        X_raw_full = X_raw_full.loc[base_mask].reset_index(drop=True); y_raw_full = y_raw_full[base_mask]; g_raw_full = g_raw_full[base_mask]; xkey_full = xkey_full[base_mask]
        rough_cols, _ = prefilter_features_groupwise(X_raw_full, y_raw_full, g_raw_full, top_k=get_prefilter_topk_for_output_local(output_name), corr_threshold=CORR_PRUNE_THRESHOLD, random_state=RANDOM_STATE+999)
        if len(rough_cols) < 2:
            raise RuntimeError("rough_cols < 2 in refit")
        selected_train_df, candidate_df = select_best_y_within_group(X_raw_full, y_raw_full, g_raw_full, xkey_full, rough_cols=rough_cols, strategy=chosen_strategy, random_state=RANDOM_STATE+999)
        selected_train_df = _ensure_standard_cols_refit(selected_train_df, group_col=group_col, xkey_col=xkey_col, group_values=g_raw_full, xkey_values=xkey_full)
        selected_train_df = selected_train_df.dropna(subset=["target","GROUP_ID"]).reset_index(drop=True)
        if selected_train_df.empty:
            raise RuntimeError("selected_train_df empty after cleanup")
        if "XKEY" not in selected_train_df.columns:
            selected_train_df["XKEY"] = [f"FULL_{i}" for i in range(len(selected_train_df))]
        X_train_sel = selected_train_df[feature_cols].copy(); y_train_sel = selected_train_df["target"].to_numpy(dtype=float); g_train_sel = selected_train_df["GROUP_ID"].astype(str).to_numpy()
        ranked_features, rank_df = build_ranked_features_train(X_train_sel, y_train_sel, g_train_sel, output_name, 999)
        if len(ranked_features) < 2:
            raise RuntimeError("ranked_features < 2 in refit")
        res = evaluate_method_train_test(chosen_method, output_name, X_train_sel, y_train_sel, g_train_sel, selected_train_df, X_train_sel.copy(), y_train_sel.copy(), ranked_features, random_state=RANDOM_STATE+999)
        if res is None:
            raise RuntimeError("evaluate_method_train_test returned None in refit")
        feats = list(res["selected_features"]); fitted_estimator = res["fitted_estimator"]
        if chosen_method in ("spca_ridge","spca_huber","spca_pls"):
            Z_full = transform_spca(fitted_estimator["spca_obj"], X_train_sel[ranked_features]); train_pred = np.asarray(fitted_estimator["reg_model"].predict(Z_full)).reshape(-1)
        elif chosen_method == "block_pca_ridge":
            Z_full = transform_block_pca(fitted_estimator["block_obj"], X_train_sel); train_pred = np.asarray(fitted_estimator["reg_model"].predict(Z_full)).reshape(-1)
        elif chosen_method == "minimal_class_average":
            members = fitted_estimator["members"]; weights = np.asarray(fitted_estimator["weights"], dtype=float); preds=[]
            for mem in members:
                preds.append(np.asarray(mem["est"].predict(X_train_sel[mem["feats"]])).reshape(-1))
            train_pred = np.average(np.vstack(preds), axis=0, weights=weights)
        elif chosen_method == "direct_omp_ridge":
            prep=fitted_estimator["prep"]; ridge=fitted_estimator["ridge"]; idx=fitted_estimator["idx"]; train_pred = np.asarray(ridge.predict(prep.transform(X_train_sel[feats])[:, idx])).reshape(-1)
        elif chosen_method in ("direct_gpr_matern","direct_gpr_rq"):
            prep=fitted_estimator["prep"]; gpr=fitted_estimator["gpr"]; train_pred=np.asarray(gpr.predict(prep.transform(X_train_sel[feats]))).reshape(-1)
        elif chosen_method in ("residual_huber_krr","residual_ard_krr"):
            if chosen_method == "residual_huber_krr":
                base_pred = np.asarray(fitted_estimator["base_estimator"].predict(X_train_sel[feats])).reshape(-1)
            else:
                base_pred = np.asarray(fitted_estimator["base_estimator"].predict(X_train_sel[feats])).reshape(-1)
            train_pred = base_pred + fitted_estimator["krr"].predict(fitted_estimator["prep"].transform(X_train_sel[feats]))
        else:
            train_pred = np.asarray(fitted_estimator.predict(X_train_sel[feats])).reshape(-1)
        tr_r2, tr_rmse, tr_mae = score_prediction(y_train_sel, train_pred)
        selected_export_cols = [c for c in ["GROUP_ID","XKEY"] + feats + ["target"] if c in selected_train_df.columns]
        selected_export_df = selected_train_df[selected_export_cols].copy().rename(columns={"target":"selected_target"})
        out_safe = sanitize_filename(output_name)
        export_df(selected_export_df, os.path.join(FINAL_DATA_DIR, f"{out_safe}_final_selected_input_output"))
        export_df(candidate_df, os.path.join(FINAL_DATA_DIR, f"{out_safe}_candidate_repeated_rows"))
        export_df(rank_df, os.path.join(FINAL_DATA_DIR, f"{out_safe}_feature_rank"))
        backup_payload = {"output_name":output_name, "method_name":chosen_method, "model_name":res["model_name"], "target_transform":res["target_transform"], "selected_features":feats, "ranked_features":ranked_features, "fitted_estimator":fitted_estimator, "selected_export_df":selected_export_df}
        backup_path = os.path.join(FINAL_BACKUP_DIR, f"{out_safe}_model_backup.joblib"); joblib.dump(backup_payload, backup_path); FINAL_MODEL_BACKUP[output_name] = backup_payload
        FINAL_REFIT_SUMMARY_ROWS.append({"output_name": output_name, "status": "ok", "reason": "", "best_y_strategy": chosen_strategy,
                                         "final_method_name": chosen_method, "final_model_name": res["model_name"], "target_transform": res["target_transform"],
                                         "final_chosen_topk": int(chosen_topk) if pd.notna(chosen_topk) else np.nan, "n_groups_full": int(len(pd.unique(g_train_sel))),
                                         "n_final_features": int(len(feats)), "final_train_r2": float(tr_r2) if pd.notna(tr_r2) else np.nan,
                                         "final_train_rmse": float(tr_rmse) if pd.notna(tr_rmse) else np.nan, "final_train_mae": float(tr_mae) if pd.notna(tr_mae) else np.nan})
    except Exception as e:
        FINAL_REFIT_SUMMARY_ROWS.append({"output_name": output_name, "status": "error", "reason": f"{type(e).__name__}: {e}",
                                         "best_y_strategy": "", "final_method_name": "", "final_model_name": "", "target_transform": "",
                                         "final_chosen_topk": np.nan, "n_groups_full": 0, "n_final_features": 0,
                                         "final_train_r2": np.nan, "final_train_rmse": np.nan, "final_train_mae": np.nan})

FINAL_REFIT_SUMMARY_DF = pd.DataFrame(FINAL_REFIT_SUMMARY_ROWS)
for c in ["output_name","status","reason","best_y_strategy","final_method_name","final_model_name","target_transform","final_chosen_topk","n_groups_full","n_final_features","final_train_r2","final_train_rmse","final_train_mae"]:
    if c not in FINAL_REFIT_SUMMARY_DF.columns:
        FINAL_REFIT_SUMMARY_DF[c] = np.nan
export_df(FINAL_REFIT_SUMMARY_DF, os.path.join(MODEL_EXPORT_DIR, "final_refit_summary"))
display(FINAL_REFIT_SUMMARY_DF.sort_values("final_train_r2", ascending=False) if not FINAL_REFIT_SUMMARY_DF.empty else FINAL_REFIT_SUMMARY_DF)


,output_name,status,reason,best_y_strategy,final_method_name,final_model_name,target_transform,final_chosen_topk,n_groups_full,n_final_features,final_train_r2,final_train_rmse,final_train_mae
13,Vibrational response | FRF (g/N) | 300-3000 Hz...,ok,,robust_trimmed_median,baseline_stability,Kernel_Ridge_RBF,yeo_johnson,4.0,128,4,0.986459,0.010487,0.004785
14,Vibrational response | FRF (g/N) | 3000-6500 H...,ok,,robust_trimmed_median,minimal_class_average,MinimalClassAverage,mixed,4.0,128,6,0.984826,0.024082,0.015316
12,Vibrational response | FRF (g/N) | 300-8000 Hz...,ok,,robust_trimmed_median,baseline_stability,Kernel_Ridge_RBF,yeo_johnson,6.0,128,6,0.951801,0.027423,0.018787
8,Thermal characteristics | h | W/m.K,ok,,singleton_raw,baseline_stability,Kernel_Ridge_RBF,yeo_johnson,4.0,25,3,0.786517,0.606806,0.470067
0,Modulus,ok,,singleton_raw,baseline_stability,Ridge,yeo_johnson,2.0,56,4,0.511028,2460.408295,1887.829743
4,Yield strength,ok,,singleton_raw,direct_ard,ARD_Regression,yeo_johnson,4.0,56,2,0.506338,33.894707,25.958719
3,AS,ok,,singleton_raw,block_pca_ridge,Ridge,yeo_johnson,11.0,56,9,0.492409,82.451519,64.379957
9,Thermal characteristics | Heating rate | °C/s,ok,,singleton_raw,stability_lasso_ridge,Ridge,yeo_johnson,3.0,65,6,0.491505,0.001630,0.001288
6,Total energy,ok,,singleton_raw,direct_theilsen,TheilSen,raw,2.0,56,2,0.434241,48.312843,38.062555
2,APS,ok,,singleton_raw,baseline_stability,Bayesian_Ridge,raw,2.0,56,2,0.386537,93.429006,72.932639
